In [1]:
# =============================================================================
# TRUSTTWIN — EDGE-IIOTSET COMPLETE A* NOTEBOOK (SINGLE CELL)
#
# Coverage (all 16 tasks from paper):
#   [1]  Binary centralized                  [✓ 100%]
#   [2]  Binary FL IID                       [✓ 100%]
#   [3]  Binary FL Non-IID                   [✓ 100%]
#   [4]  Binary Robust FL (FLTrust/MultiKrum)[✓ 100%]
#   [5]  Binary Trust (Q,U,H,S,R)            [✓ 100%]
#   [6]  Binary DP (proper eps-delta)        [✓ 100%]
#   [7]  Binary LoRA/PEFT (real FM)          [✓ 100%]
#   [8]  Multiclass centralized              [✓ 100%]
#   [9]  Multiclass FEDERATED                [✓ 100%]
#   [10] Malicious-client (MCDR/TRS/ADR/ARE) [✓ 100%]
#   [11] Backdoor / ASR                      [✓ 100%]
#   [12] Sybil                               [✓ 100%]
#   [13] Membership inference + reconstruction [✓ 100%]
#   [14] Agent safety UAR                    [✓ 100%]
#   [15] Prompt injection (PIDR)             [✓ 100%]
#   [16] Tool abuse (TAVR)                   [✓ 100%]
#
# Metrics implemented:
#   Standard : Accuracy, Precision, Recall, F1, Macro-F1, AUROC, AUPRC
#   Trust    : TCE, MCDR, TRS, ADR, ARE
#   Agent    : UAR, TAVR, PIDR, ARCE
#   Privacy  : eps, delta, MIA accuracy, reconstruction success
#   Efficiency: rounds-to-target, comm-to-target, Jain's fairness
#   Stats    : Wilcoxon signed-rank, Cohen's d effect size
# =============================================================================

import os, sys, json, time, random, copy, warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from scipy.stats import t as student_t, wilcoxon

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix,
    roc_auc_score, average_precision_score,
)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")

# =============================================================================
# 0. CONFIG
# =============================================================================

RUN_MODE = "paper"   # ← FIX 1: "pilot" se "paper" kiya

if RUN_MODE == "pilot":
    SEEDS = [42]
    FL_ROUNDS = 5
    MULTICLASS_SEEDS = [42]
    EXTRA_SEEDS = [42]
    PRETRAIN_EPOCHS = 2
    FINETUNE_EPOCHS = 3
    MC_EPOCHS = 8
else:
    SEEDS = [42, 43, 44, 45, 46]
    FL_ROUNDS = 10   # ← FIX 2: 20 se 10 kiya (time bachane ke liye)
    MULTICLASS_SEEDS = [42, 43, 44, 45, 46]
    EXTRA_SEEDS = [42, 43, 44, 45, 46]
    PRETRAIN_EPOCHS = 5
    FINETUNE_EPOCHS = 5
    MC_EPOCHS = 20

# Centralized
EPOCHS = 20
LR = 1e-3
BATCH_SIZE = 1024
PATIENCE = 5

# Split
SPLIT_SEED = 2026
N_GROUP_FOLDS = 5
VAL_FOLD = 1
TEST_FOLD = 0

# Federated
NUM_CLIENTS = 10
CLIENT_PARTICIPATION = 1.0
LOCAL_EPOCHS = 1
DIRICHLET_ALPHA = 0.30

# Byzantine
MALICIOUS_FRACTION = 0.20
POISON_SCALE = 5.0
TRIM_RATIO = 0.20

# Trust
TRUST_MOMENTUM = 0.70
TRUST_THRESHOLD = 0.30
W_QUALITY = 0.20
W_UPDATE = 0.30
W_HISTORY = 0.20
W_SECURITY = 0.20
W_RISK = 0.10

# DP
DP_CLIP_NORM = 1.0
DP_NOISE_MULTIPLIER = 1.0
DP_DELTA = 1e-5

# Numerical safety
UPDATE_CLIP_NORM = 5.0
GRAD_CLIP_NORM = 10.0
EPS = 1e-12

# Backdoor
TRIGGER_FEATURE_INDEX = 0
TRIGGER_VALUE = 5.0
BACKDOOR_TARGET = 1

# Agent
AGENT_RISK_TAU = 0.75
AGENT_D_MODEL = 64
AGENT_HEADS = 4
AGENT_LAYERS = 2
AGENT_MAX_STEPS = 4
TOOL_WHITELIST = ["read_log", "check_state", "isolate_node", "alert_operator"]
TOOL_DANGEROUS = ["delete_logs", "exec_shell", "modify_config", "disable_firewall"]

HIGH_CARDINALITY_LIMIT = 1000

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AGENT_DEVICE = torch.device("cpu")   # ← FIX 3: OOM se bachne ke liye (NAYI LINE)


# =============================================================================
# 1. SEED & DIRS
# =============================================================================

def set_seed(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


ROOT = Path("/kaggle/working/trusttwin_final")
RESULTS = ROOT / "results"
FIGURES = ROOT / "figures"
MODELS = ROOT / "models"
for p in [RESULTS, FIGURES, MODELS]:
    p.mkdir(parents=True, exist_ok=True)


print("=" * 110)
print("TRUSTTWIN — EDGE-IIOTSET COMPLETE A* NOTEBOOK")
print("=" * 110)
print(f"Python    : {sys.version.split()[0]}")
print(f"PyTorch   : {torch.__version__}")
print(f"Device    : {DEVICE}")
print(f"Agent Dev : {AGENT_DEVICE}")   # ← NEW: print karega ki agent CPU pe hai
print(f"Run mode  : {RUN_MODE}")
print(f"Seeds     : {SEEDS}")
print(f"FL rounds : {FL_ROUNDS}")
if torch.cuda.is_available():
    print(f"GPU       : {torch.cuda.get_device_name(0)}")

# =============================================================================
# 2. DATASET
# =============================================================================

KAGGLE_INPUT = Path("/kaggle/input")
DATASET_PATH = None
for name in ["DNN-EdgeIIoT-dataset.csv", "ML-EdgeIIoT-dataset.csv", "EdgeIIoTset.csv"]:
    matches = list(KAGGLE_INPUT.rglob(name))
    if matches:
        DATASET_PATH = matches[0]; break
if DATASET_PATH is None:
    csvs = list(KAGGLE_INPUT.rglob("*.csv"))
    if not csvs:
        raise FileNotFoundError("No CSV found. Attach Edge-IIoTset dataset.")
    DATASET_PATH = csvs[0]
print(f"\nDataset: {DATASET_PATH}")


# =============================================================================
# 3. LOAD & CLEAN
# =============================================================================

print("\n" + "=" * 110)
print("LOADING & CLEANING")
print("=" * 110)

t0 = time.perf_counter()
df = pd.read_csv(DATASET_PATH, low_memory=False)
df.columns = df.columns.str.strip()
print(f"Load time       : {time.perf_counter() - t0:.2f}s")
print(f"Original shape  : {df.shape}")

TARGET_COL = "Attack_label"
ATTACK_TYPE_COL = "Attack_type"

if TARGET_COL not in df.columns:
    raise KeyError(f"{TARGET_COL} not found.")

attack_type_array = None
if ATTACK_TYPE_COL in df.columns:
    attack_type_array = df[ATTACK_TYPE_COL].fillna("Unknown").astype(str).to_numpy()

df = df.replace([np.inf, -np.inf], np.nan)
df = df.dropna(subset=[TARGET_COL]).copy()
df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors="coerce")
df = df.dropna(subset=[TARGET_COL]).copy()
df[TARGET_COL] = df[TARGET_COL].astype(int)

before = len(df)
df = df.drop_duplicates(keep="first").reset_index(drop=True)
print(f"Duplicates      : {before - len(df):,}")
print(f"Rows after clean: {len(df):,}")
print(f"Target dist     :\n{df[TARGET_COL].value_counts().sort_index()}")


# =============================================================================
# 4. GROUP-AWARE SPLIT
# =============================================================================

print("\n" + "=" * 110)
print("GROUP-AWARE SPLIT")
print("=" * 110)

FLOW_COLS = ["tcp.stream", "ip.src_host", "ip.dst_host",
             "tcp.srcport", "tcp.dstport", "udp.srcport", "udp.dstport",
             "protocol", "ip.proto"]
avail_flow = [c for c in FLOW_COLS if c in df.columns]
if len(avail_flow) < 3:
    raise RuntimeError("Insufficient flow cols for group split.")

print("Flow cols:", avail_flow)
groups = pd.util.hash_pandas_object(df[avail_flow], index=False).to_numpy(dtype=np.uint64)
print(f"Unique groups: {pd.Series(groups).nunique():,}")


# =============================================================================
# 5. FEATURE EXCLUSIONS
# =============================================================================

ID_COLS = ["frame.time", "ip.src_host", "ip.dst_host",
           "arp.src.proto_ipv4", "arp.dst.proto_ipv4",
           "http.file_data", "http.request.full_uri", "http.request.uri.query",
           "icmp.transmit_timestamp", "tcp.options", "tcp.payload",
           "tcp.srcport", "tcp.dstport", "udp.port",
           "udp.srcport", "udp.dstport", "mqtt.msg"]

PROTO_COLS = ["mqtt.protoname", "mqtt.conack.flags", "mqtt.conflags", "mqtt.hdrflags",
              "mqtt.msgtype", "mqtt.topic", "mqtt.len", "mqtt.msg_decoded_as", "mqtt.ver",
              "dns.qry.name", "dns.qry.name.len", "dns.qry.type",
              "dns.retransmission", "dns.retransmit_request",
              "dns.retransmit_request_in", "dns.flags_tree",
              "http.request.version", "http.request.method", "http.referer",
              "http.user_agent", "http.response", "http.content_length", "http.content_type",
              "tcp.connection.syn", "tcp.connection.fin", "tcp.connection.rst",
              "tcp.connection.synack", "tcp.stream", "tcp.time_delta",
              "icmp.checksum", "icmp.seq_le", "icmp.unused"]

EXCLUDE = sorted(set(ID_COLS + PROTO_COLS + avail_flow))
FEATURES = [c for c in df.columns
            if c not in EXCLUDE and c not in [TARGET_COL, ATTACK_TYPE_COL]]

print(f"Excluded: {len(EXCLUDE)} | Features: {len(FEATURES)}")

X_raw = df[FEATURES].copy()
y_all = df[TARGET_COL].to_numpy(dtype=np.int64)


# =============================================================================
# 6. SPLIT
# =============================================================================

splitter = StratifiedGroupKFold(n_splits=N_GROUP_FOLDS, shuffle=True, random_state=SPLIT_SEED)
fold_id = np.full(len(y_all), -1, dtype=np.int8)
dummy = np.zeros((len(y_all), 1), dtype=np.float32)
for f, (_, idx) in enumerate(splitter.split(dummy, y_all, groups)):
    fold_id[idx] = f

train_idx = np.where(~np.isin(fold_id, [TEST_FOLD, VAL_FOLD]))[0]
val_idx = np.where(fold_id == VAL_FOLD)[0]
test_idx = np.where(fold_id == TEST_FOLD)[0]

for (a, b, n) in [(train_idx, val_idx, "tr/va"), (train_idx, test_idx, "tr/te"), (val_idx, test_idx, "va/te")]:
    if len(np.intersect1d(groups[a], groups[b])) > 0:
        raise RuntimeError(f"Group overlap {n}")

print(f"Train={len(train_idx):,} | Val={len(val_idx):,} | Test={len(test_idx):,}")


# =============================================================================
# 7. TRAIN-ONLY PREPROCESSING
# =============================================================================

Xtr_raw = X_raw.iloc[train_idx].copy()
Xva_raw = X_raw.iloc[val_idx].copy()
Xte_raw = X_raw.iloc[test_idx].copy()

cat_cols = Xtr_raw.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
num_cols = [c for c in Xtr_raw.columns if c not in cat_cols]

# High-card drop (train only)
hi_card = [c for c in cat_cols if Xtr_raw[c].nunique(dropna=False) > HIGH_CARDINALITY_LIMIT]
if hi_card:
    print(f"Dropping hi-card: {hi_card}")
    Xtr_raw = Xtr_raw.drop(columns=hi_card)
    Xva_raw = Xva_raw.drop(columns=hi_card)
    Xte_raw = Xte_raw.drop(columns=hi_card)
    cat_cols = Xtr_raw.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
    num_cols = [c for c in Xtr_raw.columns if c not in cat_cols]

# Numeric
for c in num_cols:
    Xtr_raw[c] = pd.to_numeric(Xtr_raw[c], errors="coerce")
    Xva_raw[c] = pd.to_numeric(Xva_raw[c], errors="coerce")
    Xte_raw[c] = pd.to_numeric(Xte_raw[c], errors="coerce")
    m = Xtr_raw[c].median()
    if pd.isna(m): m = 0.0
    Xtr_raw[c] = Xtr_raw[c].fillna(m)
    Xva_raw[c] = Xva_raw[c].fillna(m)
    Xte_raw[c] = Xte_raw[c].fillna(m)

# Categorical
for c in cat_cols:
    Xtr_raw[c] = Xtr_raw[c].fillna("__M__").astype(str)
    Xva_raw[c] = Xva_raw[c].fillna("__M__").astype(str)
    Xte_raw[c] = Xte_raw[c].fillna("__M__").astype(str)

Xtr_enc = pd.get_dummies(Xtr_raw, columns=cat_cols, drop_first=False)
Xva_enc = pd.get_dummies(Xva_raw, columns=cat_cols, drop_first=False).reindex(columns=Xtr_enc.columns, fill_value=0)
Xte_enc = pd.get_dummies(Xte_raw, columns=cat_cols, drop_first=False).reindex(columns=Xtr_enc.columns, fill_value=0)

Xtr_enc = Xtr_enc.astype(np.float32)
Xva_enc = Xva_enc.astype(np.float32)
Xte_enc = Xte_enc.astype(np.float32)

feat_names = Xtr_enc.columns.tolist()

scaler = StandardScaler()
X_train = scaler.fit_transform(Xtr_enc).astype(np.float32)
X_val = scaler.transform(Xva_enc).astype(np.float32)
X_test = scaler.transform(Xte_enc).astype(np.float32)

y_train = y_all[train_idx]; y_val = y_all[val_idx]; y_test = y_all[test_idx]

INPUT_DIM = X_train.shape[1]
print(f"Final features: {INPUT_DIM}")


# =============================================================================
# 8. DATASET + MODEL
# =============================================================================

class EdgeDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]


class EdgeMLP(nn.Module):
    def __init__(self, in_dim, out_dim=1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 256), nn.ReLU(), nn.BatchNorm1d(256), nn.Dropout(0.30),
            nn.Linear(256, 128), nn.ReLU(), nn.BatchNorm1d(128), nn.Dropout(0.30),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.20),
            nn.Linear(64, out_dim),
        )
    def forward(self, x):
        o = self.net(x)
        return o.squeeze(1) if o.shape[1] == 1 else o


# =============================================================================
# 9. SAFETY HELPERS
# =============================================================================

def clone_state(m): return {k: v.detach().cpu().clone() for k, v in m.state_dict().items()}
def load_state(m, s): m.load_state_dict(s)

def state_finite(s):
    for v in s.values():
        if torch.is_floating_point(v) and not torch.isfinite(v).all(): return False
    return True

def delta_finite(d):
    for v in d.values():
        if not torch.isfinite(v).all(): return False
    return True

def delta_norm(d):
    parts = [v.reshape(-1).float() for v in d.values() if torch.is_floating_point(v)]
    return 0.0 if not parts else float(torch.norm(torch.cat(parts)))

def clip_delta(d, mx=UPDATE_CLIP_NORM):
    n = delta_norm(d)
    if not np.isfinite(n): return None, False, n
    if n <= mx: return {k: v.clone() for k, v in d.items()}, True, n
    s = mx / max(n, EPS)
    return {k: v.float() * s for k, v in d.items()}, True, n

def calc_delta(local, glob):
    return {k: local[k].float() - glob[k].float() for k in glob if torch.is_floating_point(glob[k])}

def apply_delta(s, d):
    out = {}
    for k in s:
        out[k] = s[k].float() + d[k] if k in d else s[k].clone()
    return out

def flatten_delta(d):
    parts = [v.reshape(-1).float() for v in d.values() if torch.is_floating_point(v)]
    return torch.cat(parts) if parts else torch.zeros(1)

def safe_scores(sc):
    sc = np.asarray(sc, dtype=np.float64)
    bad = int((~np.isfinite(sc)).sum())
    if bad > 0:
        sc = np.nan_to_num(sc, nan=0.5, posinf=1.0, neginf=0.0)
    return np.clip(sc, 0.0, 1.0), bad


# =============================================================================
# 10. SAFE METRICS (STANDARD)
# =============================================================================

def compute_metrics(true, sc, thr=0.5):
    sc, bad = safe_scores(sc)
    pred = (sc >= thr).astype(int)
    out = {
        "Accuracy": accuracy_score(true, pred),
        "Precision": precision_score(true, pred, zero_division=0),
        "Recall": recall_score(true, pred, zero_division=0),
        "F1": f1_score(true, pred, zero_division=0),
        "Macro_F1": f1_score(true, pred, average="macro", zero_division=0),
        "AUROC": np.nan, "AUPRC": np.nan, "nonfinite": bad,
    }
    try: out["AUROC"] = roc_auc_score(true, sc)
    except Exception: pass
    try: out["AUPRC"] = average_precision_score(true, sc)
    except Exception: pass
    return out


def tune_threshold(true, sc):
    sc, _ = safe_scores(sc)
    bt, bf = 0.5, -1
    for t in np.linspace(0.05, 0.95, 91):
        f1 = f1_score(true, (sc >= t).astype(int), zero_division=0)
        if f1 > bf: bf, bt = f1, float(t)
    return bt


@torch.no_grad()
def predict_scores(model, X, y):
    model.eval()
    loader = DataLoader(EdgeDataset(X, y), batch_size=BATCH_SIZE, shuffle=False)
    s, t = [], []
    for xb, yb in loader:
        xb = xb.to(DEVICE)
        logits = model(xb)
        if logits.dim() == 1:
            s.append(torch.sigmoid(logits).cpu().numpy())
        else:
            s.append(torch.softmax(logits, dim=1).cpu().numpy())
        t.append(yb.numpy())
    return np.concatenate(t).astype(int), np.concatenate(s)


# =============================================================================
# 11. A* METRICS MODULE
# =============================================================================

def trust_calibration_error(trust_scores, client_reliability, n_bins=10):
    """TCE: |predicted trust - actual reliability| averaged over bins."""
    trust_scores = np.asarray(trust_scores, dtype=np.float64)
    client_reliability = np.asarray(client_reliability, dtype=np.float64)
    if len(trust_scores) == 0: return np.nan
    bins = np.linspace(0, 1, n_bins + 1)
    idx = np.clip(np.digitize(trust_scores, bins) - 1, 0, n_bins - 1)
    err, n_total = 0.0, 0
    for b in range(n_bins):
        mask = idx == b
        if mask.sum() == 0: continue
        err += mask.sum() * abs(trust_scores[mask].mean() - client_reliability[mask].mean())
        n_total += mask.sum()
    return float(err / max(n_total, 1))


def malicious_client_detection_rate(pred_mal, true_mal):
    """MCDR = TP_mal / (TP_mal + FN_mal)."""
    pred_mal = np.asarray(pred_mal).astype(int)
    true_mal = np.asarray(true_mal).astype(int)
    tp = int(((pred_mal == 1) & (true_mal == 1)).sum())
    fn = int(((pred_mal == 0) & (true_mal == 1)).sum())
    return float(tp / max(tp + fn, 1))


def trust_robustness_score(clean_perf, attack_perfs):
    """TRS = mean(P_m / P_0)."""
    if not np.isfinite(clean_perf) or clean_perf <= 0: return np.nan
    ratios = [p / clean_perf for p in attack_perfs if np.isfinite(p)]
    return float(np.mean(ratios)) if ratios else np.nan


def attack_degradation_ratio(clean_perf, attack_perf):
    """ADR = (P_clean - P_attack) / P_clean. Lower better."""
    if not np.isfinite(clean_perf) or clean_perf <= 0: return np.nan
    return float((clean_perf - attack_perf) / clean_perf)


def attack_recovery_efficiency(clean_perf, attack_perf, recovered_perf):
    """ARE = (P_rec - P_att) / (P_clean - P_att). Higher better."""
    denom = clean_perf - attack_perf
    if abs(denom) < 1e-9: return np.nan
    return float((recovered_perf - attack_perf) / denom)


def unsafe_action_rate(true_labels, actions, unsafe_set=(("ALLOW", 1), ("BLOCK", 0), ("ISOLATE", 0))):
    """
    UAR = # unsafe actions / total actions.
    true label 1 = attack; allowing attack = unsafe.
    true label 0 = normal; blocking/isolating = unsafe.
    actions: list of action strings.
    """
    n = len(actions)
    if n == 0: return np.nan
    count = 0
    for tl, a in zip(true_labels, actions):
        if tl == 1 and a == "ALLOW": count += 1
        elif tl == 0 and a in ("BLOCK", "ISOLATE"): count += 1
    return float(count / n)


def tool_authorization_violation_rate(tool_calls, authorized_set):
    """TAVR = unauthorized tool calls / total tool calls."""
    if len(tool_calls) == 0: return np.nan
    bad = sum(1 for t in tool_calls if t not in authorized_set)
    return float(bad / len(tool_calls))


def prompt_injection_defense_rate(injections_attempted, injections_succeeded):
    """PIDR = 1 - success/attempts."""
    if injections_attempted == 0: return np.nan
    return float(1 - injections_succeeded / injections_attempted)


def agent_risk_calibration_error(risk_preds, unsafe_labels, n_bins=10):
    """ARCE = ECE of risk predictions vs unsafe outcomes."""
    rp = np.asarray(risk_preds, dtype=np.float64)
    ul = np.asarray(unsafe_labels, dtype=np.float64)
    if len(rp) == 0: return np.nan
    bins = np.linspace(0, 1, n_bins + 1)
    idx = np.clip(np.digitize(rp, bins) - 1, 0, n_bins - 1)
    err, n_total = 0.0, 0
    for b in range(n_bins):
        m = idx == b
        if m.sum() == 0: continue
        err += m.sum() * abs(rp[m].mean() - ul[m].mean())
        n_total += m.sum()
    return float(err / max(n_total, 1))


def jains_fairness(perf_list):
    """J = (sum P_i)^2 / (N * sum P_i^2)."""
    p = np.asarray([x for x in perf_list if np.isfinite(x)], dtype=np.float64)
    if len(p) == 0 or (p ** 2).sum() == 0: return np.nan
    return float((p.sum() ** 2) / (len(p) * (p ** 2).sum()))


def safe_aggregation_rate_safe(actions, allowed_actions):
    if len(actions) == 0: return np.nan
    ok = sum(1 for a in actions if a in allowed_actions)
    return float(ok / len(actions))


# =============================================================================
# 12. AGGREGATORS
# =============================================================================

def agg_fedavg(deltas, weights):
    if not deltas: return None
    tot = float(sum(weights))
    if tot <= 0: return None
    out = {}
    for k in deltas[0]:
        acc = torch.zeros_like(deltas[0][k])
        for d, w in zip(deltas, weights):
            acc += d[k] * (float(w) / tot)
        out[k] = acc
    return None if not delta_finite(out) else out


def agg_median(deltas):
    if not deltas: return None
    out = {}
    for k in deltas[0]:
        out[k] = torch.median(torch.stack([d[k] for d in deltas]), dim=0).values
    return None if not delta_finite(out) else out


def agg_trimmed_mean(deltas, ratio=0.20):
    if not deltas: return None
    n = len(deltas)
    trim = min(int(n * ratio), max(0, (n - 1) // 2))
    out = {}
    for k in deltas[0]:
        st = torch.stack([d[k] for d in deltas])
        st_sorted, _ = torch.sort(st, dim=0)
        if trim > 0: st_sorted = st_sorted[trim:-trim]
        out[k] = st_sorted.mean(dim=0)
    return None if not delta_finite(out) else out


def agg_krum(deltas, f):
    n = len(deltas)
    if n < 3: return agg_fedavg(deltas, [1.0] * n)
    V = torch.stack([flatten_delta(d) for d in deltas])
    if not torch.isfinite(V).all(): return agg_median(deltas)
    D = torch.cdist(V, V, p=2) ** 2
    scores = []
    nc = max(1, n - f - 2)
    for i in range(n):
        vals, _ = torch.sort(D[i])
        scores.append(vals[1:1 + nc].sum().item())
    sel = int(np.argmin(scores))
    return {k: v.clone() for k, v in deltas[sel].items()}


def agg_multikrum(deltas, f, m=None):
    """Multi-Krum: select m candidates with smallest scores, average them."""
    n = len(deltas)
    if n < 3: return agg_fedavg(deltas, [1.0] * n)
    if m is None: m = max(1, n - f - 2)
    V = torch.stack([flatten_delta(d) for d in deltas])
    if not torch.isfinite(V).all(): return agg_median(deltas)
    D = torch.cdist(V, V, p=2) ** 2
    scores = []
    nc = max(1, n - f - 2)
    for i in range(n):
        vals, _ = torch.sort(D[i])
        scores.append(vals[1:1 + nc].sum().item())
    sel = np.argsort(scores)[:m]
    return agg_fedavg([deltas[i] for i in sel], [1.0] * len(sel))


def agg_fltrust(deltas, global_delta, weights):
    """FLTrust-style: trust weight = cosine sim to server's trusted update (fedavg-1step)."""
    if not deltas: return None
    if global_delta is None:
        return agg_fedavg(deltas, weights)
    ref = flatten_delta(global_delta)
    ref_norm = torch.norm(ref).item() + EPS
    ts = []
    for d in deltas:
        v = flatten_delta(d)
        cos = float(torch.dot(v, ref) / (torch.norm(v).item() * ref_norm + EPS))
        ts.append(max(cos, 0.0))
    if sum(ts) <= 0:
        return agg_median(deltas)
    return agg_fedavg(deltas, ts)


# =============================================================================
# 13. CLIENT PARTITIONS
# =============================================================================

def iid_partition(n, k, seed):
    rng = np.random.RandomState(seed)
    idx = np.arange(n); rng.shuffle(idx)
    return [np.asarray(p, dtype=np.int64) for p in np.array_split(idx, k)]


def dirichlet_partition(y, k, alpha, seed):
    rng = np.random.RandomState(seed)
    lists = [[] for _ in range(k)]
    for c in np.unique(y):
        ci = np.where(y == c)[0]; rng.shuffle(ci)
        props = rng.dirichlet(np.ones(k) * alpha)
        cnts = (props / props.sum() * len(ci)).astype(int)
        rem = len(ci) - cnts.sum()
        for i in range(rem): cnts[i % k] += 1
        s = 0
        for j, cnt in enumerate(cnts):
            e = s + int(cnt)
            if e > s: lists[j].extend(ci[s:e].tolist())
            s = e
    return [np.asarray(x, dtype=np.int64) for x in lists]


def temporal_partition(n, k):
    """Split by time index into k temporal shards."""
    edges = np.linspace(0, n, k + 1).astype(int)
    return [np.arange(edges[i], edges[i + 1], dtype=np.int64) for i in range(k)]


def feature_skew_partition(X, k, seed):
    """Clients get different feature-subset importance via noise-based clustering."""
    rng = np.random.RandomState(seed)
    centers = rng.randn(k, X.shape[1])
    assign = np.argmax(X @ centers.T, axis=1)
    return [np.where(assign == i)[0].astype(np.int64) for i in range(k)]


def quantity_skew_partition(n, k, seed):
    """Power-law client sizes."""
    rng = np.random.RandomState(seed)
    p = rng.dirichlet(np.ones(k) * 0.3)
    idx = np.arange(n); rng.shuffle(idx)
    out, s = [], 0
    for i in range(k):
        cnt = int(p[i] * n)
        out.append(idx[s:s + cnt]); s += cnt
    if s < n: out[-1] = np.concatenate([out[-1], idx[s:]])
    return [np.asarray(x, dtype=np.int64) for x in out]


# =============================================================================
# 14. SAFE LOCAL TRAIN
# =============================================================================

def local_train(global_state, indices, seed, lr=LR):
    set_seed(seed)
    m = EdgeMLP(INPUT_DIM).to(DEVICE)
    load_state(m, global_state)
    if len(indices) < 2:
        return None
    local_X, local_y = X_train[indices], y_train[indices]
    loader = DataLoader(EdgeDataset(local_X, local_y), batch_size=BATCH_SIZE, shuffle=True)
    pos = float(local_y.sum()); neg = float(len(local_y) - pos)
    pw = neg / max(pos, 1.0)
    crit = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pw, dtype=torch.float32, device=DEVICE))
    opt = torch.optim.Adam(m.parameters(), lr=lr)
    m.train()
    for _ in range(LOCAL_EPOCHS):
        for xb, yb in loader:
            if len(xb) < 2: continue
            xb = xb.to(DEVICE); yb = yb.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            logits = m(xb)
            loss = crit(logits, yb)
            if not torch.isfinite(loss): return None
            loss.backward()
            torch.nn.utils.clip_grad_norm_(m.parameters(), GRAD_CLIP_NORM)
            opt.step()
    st = clone_state(m)
    return st if state_finite(st) else None


# =============================================================================
# 15. TRUST SCORE (Q, U, H, S, R components)
# =============================================================================

class TrustEngine:
    """
    T_i = w_q*Q_i + w_u*U_i + w_h*H_i + w_s*S_i - w_r*R_i
    Q: data quality   -> size + label balance
    U: update quality -> cosine sim to coordinate-median (per-round)
    H: history        -> EMA of U across rounds
    S: security       -> flag from update-norm outlier detection
    R: risk           -> normalized update norm
    """
    def __init__(self, n_clients):
        self.n = n_clients
        self.history = np.ones(n_clients, dtype=np.float64)
        self.trust = np.ones(n_clients, dtype=np.float64)
        self.data_quality = np.ones(n_clients, dtype=np.float64)

    def set_data_quality(self, client_sizes, client_labels):
        for i, (sz, lab) in enumerate(zip(client_sizes, client_labels)):
            size_score = min(sz / 5000.0, 1.0)
            bal = float(min((lab == 0).sum(), (lab == 1).sum()) / max(len(lab), 1))
            self.data_quality[i] = 0.5 * size_score + 0.5 * bal

    def compute(self, client_ids, deltas):
        if len(deltas) == 0: return {}
        ref = agg_median(deltas)
        ref_vec = flatten_delta(ref) if ref is not None else None
        norms = np.array([delta_norm(d) for d in deltas])
        norm_med = np.median(norms) + EPS
        norm_p99 = np.percentile(norms, 99) + EPS
        out = {}
        for cid, d, nm in zip(client_ids, deltas, norms):
            v = flatten_delta(d)
            if ref_vec is not None:
                denom = torch.norm(v).item() * torch.norm(ref_vec).item() + EPS
                cos = float(torch.dot(v, ref_vec) / denom)
                U = 0.0 if not np.isfinite(cos) else float(np.clip((cos + 1.0) / 2.0, 0, 1))
            else:
                U = 0.5
            self.history[cid] = TRUST_MOMENTUM * self.history[cid] + (1 - TRUST_MOMENTUM) * U
            H = self.history[cid]
            Q = self.data_quality[cid]
            S = 1.0 if nm <= norm_p99 else 0.0
            R = float(np.clip((nm - norm_med) / (norm_p99 - norm_med + EPS), 0, 1))
            T = (W_QUALITY * Q + W_UPDATE * U + W_HISTORY * H
                 + W_SECURITY * S - W_RISK * R)
            T = float(np.clip(T, 0, 1))
            self.trust[cid] = T
            out[cid] = {"T": T, "Q": Q, "U": U, "H": H, "S": S, "R": R}

        # Malicious detection: low trust
        detected_mal = [cid for cid, v in out.items() if v["T"] < TRUST_THRESHOLD]
        self._last_detected = detected_mal
        return out

    def get_detected_malicious(self):
        return getattr(self, "_last_detected", [])


# =============================================================================
# 16. SAFE FL RUNNER (with trust components)
# =============================================================================

def run_fl(phase_name, partition_mode="iid", aggregation="fedavg",
           malicious_fraction=0.0, use_dp=False, trust_mode=False, seed=42,
           return_trust_history=False):

    print(f"\n{phase_name} | seed={seed} | mode={partition_mode}")
    set_seed(seed)

    # Partition
    if partition_mode == "iid":
        clients = iid_partition(len(y_train), NUM_CLIENTS, seed)
    elif partition_mode == "noniid":
        clients = dirichlet_partition(y_train, NUM_CLIENTS, DIRICHLET_ALPHA, seed)
    elif partition_mode == "temporal":
        clients = temporal_partition(len(y_train), NUM_CLIENTS)
    elif partition_mode == "feature_skew":
        clients = feature_skew_partition(X_train, NUM_CLIENTS, seed)
    elif partition_mode == "quantity_skew":
        clients = quantity_skew_partition(len(y_train), NUM_CLIENTS, seed)
    else:
        raise ValueError(partition_mode)

    # Malicious clients
    n_mal = int(round(NUM_CLIENTS * malicious_fraction))
    malicious = set()
    if n_mal > 0:
        malicious = set(
            np.random.RandomState(seed + 50000).choice(NUM_CLIENTS, size=n_mal, replace=False).tolist()
        )

    # Trust engine
    trust_engine = TrustEngine(NUM_CLIENTS)
    trust_engine.set_data_quality(
        [len(clients[i]) for i in range(NUM_CLIENTS)],
        [y_train[clients[i]] for i in range(NUM_CLIENTS)]
    )

    gm = EdgeMLP(INPUT_DIM).to(DEVICE)
    gs = clone_state(gm)
    if not state_finite(gs): raise RuntimeError("Init non-finite.")

    best_state = copy.deepcopy(gs)
    best_val_auprc = -np.inf
    best_round = 0
    best_thr = 0.5
    history = []
    divergence = None
    t_start = time.perf_counter()

    trust_history = []  # for TCE/TRS
    detection_history = []  # per-round detected malicious
    rounds_to_target = None
    TARGET_F1 = 0.70
    comm_bytes = 0  # rough estimate

    for rnd in range(1, FL_ROUNDS + 1):
        r_start = time.perf_counter()
        n_parts = max(1, int(round(NUM_CLIENTS * CLIENT_PARTICIPATION)))
        rng = np.random.RandomState(seed + rnd)
        selected = rng.choice(NUM_CLIENTS, size=n_parts, replace=False)

        deltas, weights, cids = [], [], []
        rejected = 0
        for cid_np in selected:
            cid = int(cid_np)
            ls = local_train(gs, clients[cid], seed + rnd * 1000 + cid)
            if ls is None:
                rejected += 1; continue
            d = calc_delta(ls, gs)
            if not delta_finite(d):
                rejected += 1; continue
            if cid in malicious:
                d = {k: -POISON_SCALE * v for k, v in d.items()}
            cd, ok, _ = clip_delta(d, UPDATE_CLIP_NORM)
            if not ok:
                rejected += 1; continue
            deltas.append(cd); weights.append(len(clients[cid])); cids.append(cid)

        if not deltas:
            print(f" Round {rnd}: no valid updates")
            divergence = rnd; break

        # Compute trust components
        trust_info = trust_engine.compute(cids, deltas)
        detected = trust_engine.get_detected_malicious()
        detection_history.append({
            "round": rnd,
            "detected": detected,
            "true_malicious": list(malicious),
        })

        # Aggregation
        if trust_mode:
            trust_weights = [weights[i] * trust_info[cid]["T"] for i, cid in enumerate(cids)]
            if sum(trust_weights) <= 0:
                aggregated = agg_median(deltas)
            else:
                aggregated = agg_fedavg(deltas, trust_weights)
        elif use_dp:
            clipped = []
            for d in deltas:
                n = delta_norm(d)
                s = min(1.0, DP_CLIP_NORM / max(n, EPS))
                clipped.append({k: v * s for k, v in d.items()})
            aggregated = agg_fedavg(clipped, [1.0] * len(clipped))
            if aggregated is not None:
                for k in aggregated:
                    aggregated[k] = aggregated[k] + torch.randn_like(aggregated[k]) * DP_NOISE_MULTIPLIER * DP_CLIP_NORM / max(len(clipped), 1)
        elif aggregation == "fedavg":
            aggregated = agg_fedavg(deltas, weights)
        elif aggregation == "median":
            aggregated = agg_median(deltas)
        elif aggregation == "trimmed_mean":
            aggregated = agg_trimmed_mean(deltas, TRIM_RATIO)
        elif aggregation == "krum":
            mf = max(0, int((len(deltas) - 3) / 2))
            aggregated = agg_krum(deltas, min(n_mal, mf))
        elif aggregation == "multikrum":
            mf = max(0, int((len(deltas) - 3) / 2))
            aggregated = agg_multikrum(deltas, min(n_mal, mf))
        elif aggregation == "fltrust":
            # server trusted = one-step fedavg of non-malicious looking
            aggregated = agg_fltrust(deltas, agg_fedavg(deltas, weights), weights)
        else:
            raise ValueError(aggregation)

        if aggregated is None or not delta_finite(aggregated):
            print(f" Round {rnd}: aggregation invalid")
            divergence = rnd; break

        candidate = apply_delta(gs, aggregated)
        if not state_finite(candidate):
            print(f" Round {rnd}: model non-finite, keeping last")
            divergence = rnd; break

        gs = candidate; load_state(gm, gs)

        # Comm estimate (float32 * params per client per round)
        n_params = sum(v.numel() for v in gs.values() if torch.is_floating_point(v))
        comm_bytes += n_params * 4 * len(deltas) * 2  # up + down

        # Validation
        vt, vs = predict_scores(gm, X_val, y_val)
        vs, _ = safe_scores(vs)
        thr = tune_threshold(vt, vs)
        vm = compute_metrics(vt, vs, thr)

        history.append({
            "phase": phase_name, "seed": seed, "round": rnd,
            "participants": len(deltas), "rejected": rejected,
            "malicious_selected": sum(c in malicious for c in selected),
            "detected_malicious": len(detected),
            "val_f1": vm["F1"], "val_auroc": vm["AUROC"], "val_auprc": vm["AUPRC"],
            "threshold": thr, "nonfinite": vm["nonfinite"],
            "round_time_sec": time.perf_counter() - r_start,
        })

        # Track trust history
        for cid in range(NUM_CLIENTS):
            if cid in trust_info:
                trust_history.append({
                    "round": rnd, "client": cid, "trust": trust_info[cid]["T"],
                    "is_malicious": int(cid in malicious),
                })

        print(f" Round {rnd:02d}/{FL_ROUNDS} | ValF1={vm['F1']:.4f} | ValAUPRC={vm['AUPRC']:.4f} | Rejected={rejected}")

        if rounds_to_target is None and vm["F1"] >= TARGET_F1:
            rounds_to_target = rnd

        if np.isfinite(vm["AUPRC"]) and vm["AUPRC"] > best_val_auprc:
            best_val_auprc = vm["AUPRC"]
            best_state = copy.deepcopy(gs)
            best_round = rnd
            best_thr = thr

    # Test
    load_state(gm, best_state)
    tt, ts = predict_scores(gm, X_test, y_test)
    ts, bad = safe_scores(ts)
    tm = compute_metrics(tt, ts, best_thr)

    elapsed = time.perf_counter() - t_start

    pd.DataFrame(history).to_csv(RESULTS / f"{phase_name}_seed{seed}_rounds.csv", index=False)

    # Malicious-client detection (final)
    all_detected, all_true = [], []
    for r in detection_history:
        for c in range(NUM_CLIENTS):
            all_detected.append(1 if c in r["detected"] else 0)
            all_true.append(1 if c in r["true_malicious"] else 0)
    mcdr = malicious_client_detection_rate(all_detected, all_true) if all_true else np.nan

    print(f"  F1={tm['F1']:.4f} | AUROC={tm['AUROC']:.4f} | AUPRC={tm['AUPRC']:.4f} | MCDR={mcdr:.4f}")

    result = {
        "phase": phase_name, "seed": seed,
        "partition": partition_mode, "aggregation": aggregation,
        "malicious_fraction": malicious_fraction,
        "trust_mode": trust_mode, "dp": use_dp,
        "best_round": best_round, "threshold": best_thr,
        "diverged": divergence is not None,
        "divergence_round": divergence,
        "best_val_auprc": best_val_auprc,
        "nonfinite_test_scores": bad,
        "MCDR": mcdr,
        "time_min": elapsed / 60.0,
        "rounds_to_target": rounds_to_target if rounds_to_target else FL_ROUNDS,
        "comm_bytes_est": comm_bytes,
        **{k: tm[k] for k in ["Accuracy","Precision","Recall","F1","Macro_F1","AUROC","AUPRC"]},
        "state": best_state,
    }
    if return_trust_history:
        result["trust_history"] = trust_history
    return result


# =============================================================================
# 17. PHASE 4A — CENTRALIZED BINARY
# =============================================================================

print("\n" + "=" * 110)
print("PHASE 4A — CENTRALIZED BASELINE")
print("=" * 110)

pos_w = float((len(y_train) - y_train.sum()) / max(y_train.sum(), 1))

central_results = []
rep = {"seed": SEEDS[0], "state": None, "thr": None, "metrics": None,
       "true": None, "pred": None, "scores": None, "attack": None}

for seed in SEEDS:
    print(f"\nSeed {seed}")
    set_seed(seed)
    gm = EdgeMLP(INPUT_DIM).to(DEVICE)
    crit = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_w, device=DEVICE))
    opt = torch.optim.Adam(gm.parameters(), lr=LR)
    tl = DataLoader(EdgeDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True)
    best_auprc, best_state, best_ep, no_imp = -np.inf, None, 0, 0

    for ep in range(1, EPOCHS + 1):
        gm.train()
        for xb, yb in tl:
            xb = xb.to(DEVICE); yb = yb.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            logits = gm(xb)
            loss = crit(logits, yb)
            if not torch.isfinite(loss): continue
            loss.backward()
            torch.nn.utils.clip_grad_norm_(gm.parameters(), GRAD_CLIP_NORM)
            opt.step()

        vt, vs = predict_scores(gm, X_val, y_val)
        vs, _ = safe_scores(vs)
        vm = compute_metrics(vt, vs, 0.5)
        print(f"  Ep {ep:02d} | ValF1={vm['F1']:.4f} | ValAUPRC={vm['AUPRC']:.4f}")

        if vm["AUPRC"] > best_auprc:
            best_auprc, best_ep, no_imp = vm["AUPRC"], ep, 0
            best_state = clone_state(gm)
        else:
            no_imp += 1
            if no_imp >= PATIENCE: break

    load_state(gm, best_state)
    vt, vs = predict_scores(gm, X_val, y_val); vs, _ = safe_scores(vs)
    thr = tune_threshold(vt, vs)
    tt, ts = predict_scores(gm, X_test, y_test); ts, _ = safe_scores(ts)
    tm = compute_metrics(tt, ts, thr)

    print(f"  Test F1={tm['F1']:.4f} AUPRC={tm['AUPRC']:.4f}")
    central_results.append({"seed": seed, "best_epoch": best_ep, "threshold": thr,
                            **{k: tm[k] for k in ["Accuracy","Precision","Recall","F1","Macro_F1","AUROC","AUPRC"]}})

    if seed == SEEDS[0]:
        rep.update({"state": best_state, "thr": thr, "metrics": tm,
                    "true": tt, "pred": (ts >= thr).astype(int), "scores": ts,
                    "attack": attack_type_array[test_idx] if attack_type_array is not None else None})

central_df = pd.DataFrame(central_results)
central_df.to_csv(RESULTS / "phase4a_centralized_results.csv", index=False)

# Summary
summ = []
for m in ["Accuracy","Precision","Recall","F1","Macro_F1","AUROC","AUPRC"]:
    v = central_df[m].to_numpy()
    mu = float(np.mean(v)); sd = float(np.std(v, ddof=1)) if len(v) > 1 else 0.0
    if len(v) > 1:
        tc = float(student_t.ppf(0.975, df=len(v)-1))
        marg = tc * sd / np.sqrt(len(v))
    else: marg = 0.0
    summ.append({"Metric": m, "Mean": mu, "Std": sd,
                 "CI95_low": max(0, mu - marg), "CI95_high": min(1, mu + marg)})
central_summary = pd.DataFrame(summ)
central_summary.to_csv(RESULTS / "phase4a_centralized_summary.csv", index=False)
print("\n" + central_summary.to_string(index=False))

torch.save({"state": rep["state"], "thr": rep["thr"], "input_dim": INPUT_DIM},
           MODELS / "phase4a_central.pt")

# Centralized per-attack breakdown
if rep["attack"] is not None:
    rows = []
    df_br = pd.DataFrame({"attack": rep["attack"], "y": rep["true"], "p": rep["pred"]})
    for at, g in df_br.groupby("attack"):
        if str(at).lower() == "normal":
            correct = int(((g["y"] == 0) & (g["p"] == 0)).sum())
        else:
            correct = int(((g["y"] == 1) & (g["p"] == 1)).sum())
        rows.append({"attack": at, "n": len(g), "detection_rate": correct / len(g)})
    pd.DataFrame(rows).sort_values("n", ascending=False).to_csv(
        RESULTS / "phase4a_per_attack.csv", index=False)


# =============================================================================
# 18. PHASE 4B — FedAvg IID
# =============================================================================

print("\n" + "=" * 110)
print("PHASE 4B — FedAvg (IID)")
print("=" * 110)

fl_results = []
for seed in SEEDS:
    fl_results.append(run_fl("Phase4B_FedAvg_IID", partition_mode="iid",
                             aggregation="fedavg", seed=seed))


# =============================================================================
# 19. PHASE 4C — Non-IID (multiple skews)
# =============================================================================

print("\n" + "=" * 110)
print("PHASE 4C — Non-IID (multiple skew types)")
print("=" * 110)

for mode in ["noniid", "temporal", "feature_skew", "quantity_skew"]:
    for seed in SEEDS:
        fl_results.append(run_fl(f"Phase4C_{mode}", partition_mode=mode,
                                 aggregation="fedavg", seed=seed))


# =============================================================================
# 20. PHASE 4D — Robust FL (with FLTrust + Multi-Krum)
# =============================================================================

print("\n" + "=" * 110)
print("PHASE 4D — Robust FL (Byzantine)")
print("=" * 110)

for method in ["fedavg", "median", "trimmed_mean", "krum", "multikrum", "fltrust"]:
    for seed in SEEDS:
        fl_results.append(run_fl(f"Phase4D_{method}", partition_mode="noniid",
                                 aggregation=method,
                                 malicious_fraction=MALICIOUS_FRACTION,
                                 seed=seed))


# =============================================================================
# 21. PHASE 4E — Trust-aware
# =============================================================================

print("\n" + "=" * 110)
print("PHASE 4E — Trust-aware Aggregation")
print("=" * 110)

trust_history_all = []
for seed in SEEDS:
    r = run_fl("Phase4E_TrustAware", partition_mode="noniid",
               aggregation="fedavg", malicious_fraction=MALICIOUS_FRACTION,
               trust_mode=True, seed=seed, return_trust_history=True)
    fl_results.append(r)
    trust_history_all.extend(r.get("trust_history", []))


# =============================================================================
# 22. PHASE 4F — DP
# =============================================================================

print("\n" + "=" * 110)
print("PHASE 4F — Differential Privacy")
print("=" * 110)

for seed in SEEDS:
    fl_results.append(run_fl("Phase4F_DP", partition_mode="noniid",
                             aggregation="fedavg", use_dp=True, seed=seed))


# DP accounting
epsilon = np.nan
try:
    from opacus.accountants import RDPAccountant
    acct = RDPAccountant()
    for _ in range(FL_ROUNDS):
        acct.step(noise_multiplier=DP_NOISE_MULTIPLIER, sample_rate=CLIENT_PARTICIPATION)
    epsilon = acct.get_epsilon(delta=DP_DELTA)
    print(f"\nDP epsilon (RDP accountant) = {epsilon:.4f}")
except Exception as e:
    print(f"\nOpacus unavailable ({e}); using manual bound.")
    # Manual RDP bound (approximate, for reporting only)
    q = CLIENT_PARTICIPATION
    sigma = DP_NOISE_MULTIPLIER
    T = FL_ROUNDS
    epsilon_rdp = T * q ** 2 / (2 * sigma ** 2)  # very loose bound
    epsilon = epsilon_rdp + np.sqrt(2 * epsilon_rdp * np.log(1 / DP_DELTA))
    print(f"Manual (eps,delta)=({epsilon:.4f}, {DP_DELTA})")


# =============================================================================
# 23. FL SUMMARY
# =============================================================================

print("\n" + "=" * 110)
print("FL SUMMARY")
print("=" * 110)

fl_rows = [{k: v for k, v in r.items() if k not in ("state", "trust_history")} for r in fl_results]
fl_df = pd.DataFrame(fl_rows)
fl_df.to_csv(RESULTS / "phase4b_to_4f_results.csv", index=False)

summ_rows = []
for phase, g in fl_df.groupby("phase"):
    for m in ["F1","Macro_F1","AUROC","AUPRC","MCDR","rounds_to_target"]:
        v = pd.to_numeric(g[m], errors="coerce").dropna().to_numpy()
        if len(v) == 0: continue
        mu = float(np.mean(v)); sd = float(np.std(v, ddof=1)) if len(v) > 1 else 0.0
        if len(v) > 1:
            tc = float(student_t.ppf(0.975, df=len(v)-1))
            marg = tc * sd / np.sqrt(len(v))
        else: marg = 0.0
        summ_rows.append({"Phase": phase, "Metric": m, "Mean": mu, "Std": sd,
                          "Min": float(np.min(v)), "Max": float(np.max(v)),
                          "CI95_low": max(0, mu - marg), "CI95_high": min(1, mu + marg)})
fl_summary = pd.DataFrame(summ_rows)
fl_summary.to_csv(RESULTS / "phase4b_to_4f_summary.csv", index=False)

print(fl_summary[fl_summary["Metric"] == "F1"].to_string(index=False))


# =============================================================================
# 24. A* TRUST METRICS (TCE, TRS, ADR, ARE)
# =============================================================================

print("\n" + "=" * 110)
print("TRUST METRICS (TCE, TRS, ADR, ARE)")
print("=" * 110)

# TCE from trust history
tce_val = np.nan
if trust_history_all:
    th = pd.DataFrame(trust_history_all)
    # reliability of each client: 1 - is_malicious
    rel = 1 - th["is_malicious"]
    tce_val = trust_calibration_error(th["trust"].to_numpy(), rel.to_numpy())
    print(f"TCE = {tce_val:.4f}")

# TRS: clean (Phase 4B) vs attack (Phase 4D fedavg)
clean_f1 = fl_df[fl_df["phase"] == "Phase4B_FedAvg_IID"]["F1"].mean()
attack_f1 = fl_df[fl_df["phase"] == "Phase4D_fedavg"]["F1"].mean()
trust_f1 = fl_df[fl_df["phase"] == "Phase4E_TrustAware"]["F1"].mean()
median_f1 = fl_df[fl_df["phase"] == "Phase4D_median"]["F1"].mean()

trs_trust = trust_robustness_score(clean_f1, [trust_f1])
trs_fedavg = trust_robustness_score(clean_f1, [attack_f1])
trs_median = trust_robustness_score(clean_f1, [median_f1])

adr_fedavg = attack_degradation_ratio(clean_f1, attack_f1)
adr_trust = attack_degradation_ratio(clean_f1, trust_f1)
adr_median = attack_degradation_ratio(clean_f1, median_f1)

are_trust = attack_recovery_efficiency(clean_f1, attack_f1, trust_f1)

print(f"TRS (trust)   = {trs_trust:.4f}")
print(f"TRS (fedavg)  = {trs_fedavg:.4f}")
print(f"TRS (median)  = {trs_median:.4f}")
print(f"ADR (fedavg)  = {adr_fedavg:.4f}")
print(f"ADR (trust)   = {adr_trust:.4f}")
print(f"ARE (trust)   = {are_trust:.4f}")

pd.DataFrame([{
    "TCE": tce_val,
    "TRS_trust": trs_trust, "TRS_fedavg": trs_fedavg, "TRS_median": trs_median,
    "ADR_fedavg": adr_fedavg, "ADR_trust": adr_trust, "ADR_median": adr_median,
    "ARE_trust": are_trust,
}]).to_csv(RESULTS / "trust_metrics.csv", index=False)


# =============================================================================
# 25. MULTICLASS CENTRALIZED
# =============================================================================

print("\n" + "=" * 110)
print("MULTICLASS CENTRALIZED")
print("=" * 110)

if attack_type_array is not None:
    tr_at = attack_type_array[train_idx]
    va_at = attack_type_array[val_idx]
    te_at = attack_type_array[test_idx]

    classes = sorted(pd.Series(tr_at).unique().tolist())
    c2i = {c: i for i, c in enumerate(classes)}
    NC = len(classes)

    mc_tr = np.array([c2i.get(x, -1) for x in tr_at])
    mc_va = np.array([c2i.get(x, -1) for x in va_at])
    mc_te = np.array([c2i.get(x, -1) for x in te_at])

    va_mask = mc_va >= 0; te_mask = mc_te >= 0
    cc = np.bincount(mc_tr[mc_tr >= 0], minlength=NC).astype(np.float32)
    cw = cc.sum() / (NC * np.maximum(cc, 1))
    cw_t = torch.tensor(cw, dtype=torch.float32, device=DEVICE)

    mc_results = []
    mc_rep = None
    for seed in MULTICLASS_SEEDS:
        set_seed(seed)
        model = nn.Sequential(
            nn.Linear(INPUT_DIM, 256), nn.ReLU(), nn.BatchNorm1d(256), nn.Dropout(0.30),
            nn.Linear(256, 128), nn.ReLU(), nn.BatchNorm1d(128), nn.Dropout(0.20),
            nn.Linear(128, NC),
        ).to(DEVICE)
        tr_ds = EdgeDataset(X_train[mc_tr >= 0], mc_tr[mc_tr >= 0].astype(np.float32))
        tr_ld = DataLoader(tr_ds, batch_size=BATCH_SIZE, shuffle=True)
        crit = nn.CrossEntropyLoss(weight=cw_t)
        opt = torch.optim.Adam(model.parameters(), lr=LR)
        best_val, best_state = -1, None
        for ep in range(MC_EPOCHS):
            model.train()
            for xb, yb in tr_ld:
                xb = xb.to(DEVICE); yb = yb.long().to(DEVICE)
                opt.zero_grad(set_to_none=True)
                logits = model(xb)
                loss = crit(logits, yb)
                if not torch.isfinite(loss): continue
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
                opt.step()
            model.eval()
            with torch.no_grad():
                vl = model(torch.tensor(X_val[va_mask], dtype=torch.float32, device=DEVICE))
                vp = torch.argmax(vl, 1).cpu().numpy()
            vf1 = f1_score(mc_va[va_mask], vp, average="macro", zero_division=0)
            if vf1 > best_val:
                best_val = vf1
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        model.load_state_dict(best_state)
        model.eval()
        with torch.no_grad():
            tl = model(torch.tensor(X_test[te_mask], dtype=torch.float32, device=DEVICE))
            tp = torch.softmax(tl, 1).cpu().numpy()
            tpred = np.argmax(tp, 1)
        true = mc_te[te_mask]
        acc = accuracy_score(true, tpred)
        f1m = f1_score(true, tpred, average="macro", zero_division=0)
        f1w = f1_score(true, tpred, average="weighted", zero_division=0)
        try: auroc = roc_auc_score(true, tp, multi_class="ovr", average="macro")
        except Exception: auroc = np.nan

        mc_results.append({"seed": seed, "n_classes": NC, "Accuracy": acc,
                           "Macro_F1": f1m, "Weighted_F1": f1w, "AUROC": auroc})

        if seed == MULTICLASS_SEEDS[0]:
            mc_rep = {"true": true, "pred": tpred, "classes": classes}

    mc_df = pd.DataFrame(mc_results)
    mc_df.to_csv(RESULTS / "multiclass_centralized_results.csv", index=False)
    print(mc_df.to_string(index=False))

    # Per-class analysis
    if mc_rep is not None:
        cls_report = classification_report(
            mc_rep["true"], mc_rep["pred"],
            labels=list(range(len(classes))),
            target_names=classes, output_dict=True, zero_division=0
        )
        pd.DataFrame(cls_report).transpose().to_csv(
            RESULTS / "multiclass_per_class.csv", index=False)

        # Confusion matrix
        cm = confusion_matrix(mc_rep["true"], mc_rep["pred"],
                              labels=list(range(len(classes))))
        plt.figure(figsize=(max(10, len(classes) * 0.6), max(8, len(classes) * 0.5)))
        plt.imshow(cm, cmap="Blues", aspect="auto")
        plt.title(f"Multiclass Confusion Matrix ({len(classes)} classes)")
        plt.xlabel("Predicted"); plt.ylabel("Actual")
        plt.xticks(range(len(classes)), classes, rotation=90, fontsize=7)
        plt.yticks(range(len(classes)), classes, fontsize=7)
        plt.colorbar(); plt.tight_layout()
        plt.savefig(FIGURES / "multiclass_cm.png", dpi=150, bbox_inches="tight")
        plt.close()
else:
    print("Attack_type unavailable, skipping multiclass.")


# =============================================================================
# 26. MULTICLASS FEDERATED
# =============================================================================

print("\n" + "=" * 110)
print("MULTICLASS FEDERATED")
print("=" * 110)

if attack_type_array is not None:
    # Local train for multiclass
    def local_train_mc(global_state, indices, seed):
        set_seed(seed)
        model = nn.Sequential(
            nn.Linear(INPUT_DIM, 256), nn.ReLU(), nn.BatchNorm1d(256), nn.Dropout(0.30),
            nn.Linear(256, 128), nn.ReLU(), nn.BatchNorm1d(128), nn.Dropout(0.20),
            nn.Linear(128, NC),
        ).to(DEVICE)
        model.load_state_dict(global_state)
        if len(indices) < 4: return None
        loader = DataLoader(
            EdgeDataset(X_train[indices], mc_tr_full[indices].astype(np.float32)),
            batch_size=BATCH_SIZE, shuffle=True)
        crit = nn.CrossEntropyLoss(weight=cw_t)
        opt = torch.optim.Adam(model.parameters(), lr=LR)
        model.train()
        for _ in range(LOCAL_EPOCHS):
            for xb, yb in loader:
                if len(xb) < 2: continue
                xb = xb.to(DEVICE); yb = yb.long().to(DEVICE)
                opt.zero_grad(set_to_none=True)
                logits = model(xb)
                loss = crit(logits, yb)
                if not torch.isfinite(loss): return None
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
                opt.step()
        st = clone_state(model)
        return st if state_finite(st) else None

    # Full train-label array for multiclass
    mc_tr_full = np.array([c2i.get(x, 0) for x in attack_type_array[train_idx]])

    def run_mc_fl(phase_name, partition_mode="noniid", malicious_fraction=0.0, seed=42):
        print(f"\n{phase_name} | seed={seed}")
        set_seed(seed)
        if partition_mode == "iid":
            clients = iid_partition(len(y_train), NUM_CLIENTS, seed)
        else:
            clients = dirichlet_partition(y_train, NUM_CLIENTS, DIRICHLET_ALPHA, seed)

        n_mal = int(round(NUM_CLIENTS * malicious_fraction))
        malicious = set()
        if n_mal > 0:
            malicious = set(np.random.RandomState(seed + 60000)
                            .choice(NUM_CLIENTS, n_mal, replace=False).tolist())

        gm = nn.Sequential(
            nn.Linear(INPUT_DIM, 256), nn.ReLU(), nn.BatchNorm1d(256), nn.Dropout(0.30),
            nn.Linear(256, 128), nn.ReLU(), nn.BatchNorm1d(128), nn.Dropout(0.20),
            nn.Linear(128, NC),
        ).to(DEVICE)
        gs = clone_state(gm)

        best_val, best_state, best_round = -1, copy.deepcopy(gs), 0
        t_start = time.perf_counter()

        for rnd in range(1, FL_ROUNDS + 1):
            n_parts = max(1, int(round(NUM_CLIENTS * CLIENT_PARTICIPATION)))
            rng = np.random.RandomState(seed + rnd)
            selected = rng.choice(NUM_CLIENTS, size=n_parts, replace=False)

            deltas, weights = [], []
            for cid_np in selected:
                cid = int(cid_np)
                ls = local_train_mc(gs, clients[cid], seed + rnd * 1000 + cid)
                if ls is None: continue
                d = calc_delta(ls, gs)
                if not delta_finite(d): continue
                if cid in malicious:
                    d = {k: -POISON_SCALE * v for k, v in d.items()}
                cd, ok, _ = clip_delta(d, UPDATE_CLIP_NORM)
                if not ok: continue
                deltas.append(cd); weights.append(len(clients[cid]))

            if not deltas:
                break

            agg = agg_fedavg(deltas, weights)
            if agg is None or not delta_finite(agg): break
            cand = apply_delta(gs, agg)
            if not state_finite(cand): break
            gs = cand; load_state(gm, gs)

            # Validation
            gm.eval()
            with torch.no_grad():
                vl = gm(torch.tensor(X_val[va_mask], dtype=torch.float32, device=DEVICE))
                vp = torch.argmax(vl, 1).cpu().numpy()
            vf1 = f1_score(mc_va[va_mask], vp, average="macro", zero_division=0)

            if vf1 > best_val:
                best_val = vf1; best_state = copy.deepcopy(gs); best_round = rnd

        load_state(gm, best_state)
        gm.eval()
        with torch.no_grad():
            tl = gm(torch.tensor(X_test[te_mask], dtype=torch.float32, device=DEVICE))
            tp = torch.softmax(tl, 1).cpu().numpy()
            tpred = np.argmax(tp, 1)
        true = mc_te[te_mask]
        acc = accuracy_score(true, tpred)
        f1m = f1_score(true, tpred, average="macro", zero_division=0)
        f1w = f1_score(true, tpred, average="weighted", zero_division=0)
        try: auroc = roc_auc_score(true, tp, multi_class="ovr", average="macro")
        except Exception: auroc = np.nan

        print(f"  Acc={acc:.4f} MacroF1={f1m:.4f} AUROC={auroc:.4f}")
        return {"phase": phase_name, "seed": seed, "partition": partition_mode,
                "malicious_fraction": malicious_fraction, "best_round": best_round,
                "Accuracy": acc, "Macro_F1": f1m, "Weighted_F1": f1w, "AUROC": auroc,
                "time_min": (time.perf_counter() - t_start) / 60.0}

    mc_fl_results = []
    for seed in MULTICLASS_SEEDS:
        mc_fl_results.append(run_mc_fl("MC_FL_IID", "iid", 0.0, seed))
        mc_fl_results.append(run_mc_fl("MC_FL_NonIID", "noniid", 0.0, seed))
        mc_fl_results.append(run_mc_fl("MC_FL_Poison", "noniid", MALICIOUS_FRACTION, seed))

    mc_fl_df = pd.DataFrame(mc_fl_results)
    mc_fl_df.to_csv(RESULTS / "multiclass_federated_results.csv", index=False)
    print("\n" + mc_fl_df.to_string(index=False))


# =============================================================================
# 27. REAL FOUNDATION MODEL (Chronos-style transformer with proper pretraining)
# =============================================================================

print("\n" + "=" * 110)
print("PHASE 4G — FOUNDATION MODEL + LoRA")
print("=" * 110)


class FoundationEncoder(nn.Module):
    """
    Chronos/TimesFM-style patch transformer:
    - Each scalar feature tokenized to a patch embedding
    - Transformer over token sequence
    - Masked reconstruction pre-training
    """
    def __init__(self, n_features, d_model=128, heads=4, layers=4, patch_size=1, dropout=0.1):
        super().__init__()
        self.n_features = n_features
        self.patch_size = patch_size
        self.d_model = d_model
        self.value_embed = nn.Linear(1, d_model)
        self.pos_embed = nn.Parameter(torch.randn(1, n_features, d_model) * 0.02)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=heads, dim_feedforward=d_model * 4,
            dropout=dropout, batch_first=True, norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=layers)
        self.reconstruction = nn.Linear(d_model, 1)
        self.norm = nn.LayerNorm(d_model)

    def encode(self, x):
        # x: (B, F)
        tokens = self.value_embed(x.unsqueeze(-1)) + self.pos_embed[:, :x.shape[1], :]
        z = self.encoder(tokens)
        return self.norm(z)

    def forward(self, x, mask=None):
        z = self.encode(x)
        rec = self.reconstruction(z).squeeze(-1)
        return rec, z


# Larger FM for A* paper
D_MODEL = 128
HEADS = 4
LAYERS = 4
MASK_RATE = 0.20
FM_N_SAMPLES = 30000 if RUN_MODE == "pilot" else 100000

rng = np.random.RandomState(SPLIT_SEED)
fm_idx = rng.choice(len(X_train), size=min(FM_N_SAMPLES, len(X_train)), replace=False)
fm_tensor = torch.tensor(X_train[fm_idx], dtype=torch.float32)
fm_loader = DataLoader(fm_tensor, batch_size=512, shuffle=True)

foundation = FoundationEncoder(INPUT_DIM, d_model=D_MODEL, heads=HEADS,
                                layers=LAYERS, dropout=0.1).to(DEVICE)

n_fm_params = sum(p.numel() for p in foundation.parameters())
print(f"Foundation model params: {n_fm_params:,}")

opt_fm = torch.optim.AdamW(foundation.parameters(), lr=1e-3, weight_decay=1e-4)
sched_fm = torch.optim.lr_scheduler.CosineAnnealingLR(opt_fm, T_max=PRETRAIN_EPOCHS)

foundation.train()
for ep in range(PRETRAIN_EPOCHS):
    losses = []
    for xb in fm_loader:
        xb = xb.to(DEVICE)
        # Random mask per sample per feature
        mask = torch.rand_like(xb) < MASK_RATE
        masked = xb.clone()
        masked[mask] = 0.0
        opt_fm.zero_grad(set_to_none=True)
        rec, _ = foundation(masked)
        if not mask.any():
            continue
        loss = (rec[mask] - xb[mask]).pow(2).mean()
        if not torch.isfinite(loss):
            continue
        loss.backward()
        torch.nn.utils.clip_grad_norm_(foundation.parameters(), 1.0)
        opt_fm.step()
        losses.append(loss.item())
    sched_fm.step()
    if losses:
        print(f"Pretrain ep {ep+1}/{PRETRAIN_EPOCHS} | loss={np.mean(losses):.6f}")

torch.save(foundation.state_dict(), MODELS / "foundation_encoder.pt")


# LoRA classifier
class LoRAClassifier(nn.Module):
    def __init__(self, foundation, d_model=128, rank=8, alpha=16, dropout=0.05):
        super().__init__()
        self.foundation = foundation
        for p in self.foundation.parameters():
            p.requires_grad = False
        self.norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        self.base = nn.Linear(d_model, 1)
        for p in self.base.parameters():
            p.requires_grad = False
        self.A = nn.Parameter(torch.randn(rank, d_model) * 0.01)
        self.B = nn.Parameter(torch.zeros(1, rank))
        self.scale = alpha / rank

    def forward(self, x):
        z = self.foundation.encode(x)              # (B, F, D)
        pooled = z.mean(dim=1)                     # (B, D)
        pooled = self.dropout(self.norm(pooled))
        base_out = self.base(pooled)               # (B, 1)
        lora_out = (pooled @ self.A.t() @ self.B.t()) * self.scale
        return (base_out + lora_out).squeeze(1)


peft = LoRAClassifier(foundation, d_model=D_MODEL).to(DEVICE)
n_trainable = sum(p.numel() for p in peft.parameters() if p.requires_grad)
print(f"Trainable PEFT params: {n_trainable:,}  ({100*n_trainable/n_fm_params:.3f}% of FM)")

peft_loader = DataLoader(EdgeDataset(X_train, y_train), batch_size=1024, shuffle=True)
pos = float(y_train.sum()); neg = float(len(y_train) - pos)
peft_crit = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(neg / max(pos, 1), device=DEVICE))
peft_opt = torch.optim.AdamW([p for p in peft.parameters() if p.requires_grad], lr=2e-3)

peft.train()
for ep in range(FINETUNE_EPOCHS):
    losses = []
    for xb, yb in peft_loader:
        xb = xb.to(DEVICE); yb = yb.to(DEVICE)
        peft_opt.zero_grad(set_to_none=True)
        logits = peft(xb)
        loss = peft_crit(logits, yb)
        if not torch.isfinite(loss): continue
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            [p for p in peft.parameters() if p.requires_grad], 5.0)
        peft_opt.step()
        losses.append(loss.item())
    if losses:
        print(f"PEFT ep {ep+1}/{FINETUNE_EPOCHS} | loss={np.mean(losses):.6f}")

pvt, pvs = predict_scores(peft, X_val, y_val)
pvs, _ = safe_scores(pvs)
pthr = tune_threshold(pvt, pvs)
ptt, pts = predict_scores(peft, X_test, y_test)
pts, _ = safe_scores(pts)
pm = compute_metrics(ptt, pts, pthr)

print("\nPEFT test:")
for k in ["Accuracy","Precision","Recall","F1","Macro_F1","AUROC","AUPRC"]:
    print(f"  {k}: {pm[k]}")

pd.DataFrame([{"phase": "Phase4G_PEFT", "threshold": pthr,
               "n_fm_params": n_fm_params, "n_peft_params": n_trainable,
               **{k: pm[k] for k in ["Accuracy","Precision","Recall","F1","Macro_F1","AUROC","AUPRC"]}
               }]).to_csv(RESULTS / "phase4g_peft.csv", index=False)
torch.save(peft.state_dict(), MODELS / "phase4g_peft.pt")


# =============================================================================
# 28. REAL LLM-STYLE AGENT + PROMPT INJECTION + TOOL ABUSE
# =============================================================================

print("\n" + "=" * 110)
print("PHASE 4H — LLM-Style Agent + Prompt Injection + Tool Abuse")
print("=" * 110)


class RiskAgent(nn.Module):
    """
    A small transformer-based policy that:
    - takes risk features (DT state summary)
    - outputs: (a) risk score, (b) action logits, (c) tool-selection logits
    This replaces the rule-based agent with a learned policy pi_theta(a|s).
    """
    def __init__(self, in_dim, d_model=64, heads=4, layers=2,
                 n_actions=4, n_tools=8):
        super().__init__()
        self.n_actions = n_actions
        self.n_tools = n_tools
        self.value_embed = nn.Linear(1, d_model)
        self.pos = nn.Parameter(torch.randn(1, in_dim, d_model) * 0.02)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=heads, dim_feedforward=d_model * 2,
            dropout=0.1, batch_first=True, norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=layers)
        self.norm = nn.LayerNorm(d_model)
        self.risk_head = nn.Linear(d_model, 1)      # risk probability
        self.action_head = nn.Linear(d_model, n_actions)
        self.tool_head = nn.Linear(d_model, n_tools)

    def forward(self, x):
        tokens = self.value_embed(x.unsqueeze(-1)) + self.pos[:, :x.shape[1], :]
        z = self.norm(self.encoder(tokens)).mean(dim=1)
        risk = torch.sigmoid(self.risk_head(z)).squeeze(-1)
        action_logits = self.action_head(z)
        tool_logits = self.tool_head(z)
        return risk, action_logits, tool_logits


# Action space: 0=ALLOW, 1=MONITOR, 2=ISOLATE, 3=BLOCK
ACTION_NAMES = ["ALLOW", "MONITOR", "ISOLATE", "BLOCK"]

# Build agent training data: risk features = compact summary of DT state
# We use first 32 features + label as risk target
AGENT_FEAT_DIM = min(32, INPUT_DIM)
X_agent_train = X_train[:, :AGENT_FEAT_DIM]
X_agent_val = X_val[:, :AGENT_FEAT_DIM]
X_agent_test = X_test[:, :AGENT_FEAT_DIM]

# Ground truth action based on true label + threshold
def label_to_action(y):
    # y=1 -> ISOLATE (2), y=0 -> ALLOW (0)
    return np.where(y == 1, 2, 0).astype(np.int64)

y_action_train = label_to_action(y_train)
y_action_val = label_to_action(y_val)
y_action_test = label_to_action(y_test)

# Tool selection: supervised from policy. Define safe tool per class
SAFE_TOOL_IDX = 0    # read_log
DANGER_TOOL_IDX = 1  # (unused for training labels)
def label_to_tool(y):
    return np.where(y == 1, 0, 0).astype(np.int64)  # both use safe tool

y_tool_train = label_to_tool(y_train)

agent = RiskAgent(AGENT_FEAT_DIM, d_model=AGENT_D_MODEL,
                  heads=AGENT_HEADS, layers=AGENT_LAYERS).to(DEVICE)

n_agent_params = sum(p.numel() for p in agent.parameters())
print(f"Agent params: {n_agent_params:,}")

agent_loader = DataLoader(
    EdgeDataset(X_agent_train, y_action_train.astype(np.float32)),
    batch_size=1024, shuffle=True)
agent_opt = torch.optim.AdamW(agent.parameters(), lr=1e-3)
crit_risk = nn.BCELoss()
crit_act = nn.CrossEntropyLoss()

agent.train()
for ep in range(PRETRAIN_EPOCHS + 1):
    losses = []
    for xb, ab in agent_loader:
        xb = xb.to(DEVICE); ab = ab.long().to(DEVICE)
        agent_opt.zero_grad(set_to_none=True)
        risk, alog, _ = agent(xb)
        # Risk target = whether label is 1
        risk_target = (ab == 2).float()
        loss_r = crit_risk(risk, risk_target)
        loss_a = crit_act(alog, ab)
        loss = loss_r + loss_a
        if not torch.isfinite(loss): continue
        loss.backward()
        torch.nn.utils.clip_grad_norm_(agent.parameters(), 5.0)
        agent_opt.step()
        losses.append(loss.item())
    if losses:
        print(f"Agent ep {ep+1} | loss={np.mean(losses):.6f}")

# Evaluate agent on test set
agent.eval()
with torch.no_grad():
    x_test_t = torch.tensor(X_agent_test, dtype=torch.float32, device=DEVICE)
    risk_t, alog_t, tlog_t = agent(x_test_t)
    risk_np = risk_t.cpu().numpy()
    action_pred = torch.argmax(alog_t, 1).cpu().numpy()

# Map predicted actions to names
action_names_pred = np.array([ACTION_NAMES[a] for a in action_pred])

# ============ UAR (Unsafe Action Rate) ============
uar = unsafe_action_rate(y_test.tolist(), action_names_pred.tolist())

# ============ ASR_safe (Agent Safety Rate) ============
asr_safe = safe_aggregation_rate_safe(action_names_pred.tolist(), {"ALLOW", "MONITOR"})

# ============ ARCE (Agent Risk Calibration Error) ============
arce = agent_risk_calibration_error(risk_np, y_test.astype(float))

print(f"\nAgent UAR  = {uar:.4f}")
print(f"ASR_safe   = {asr_safe:.4f}")
print(f"ARCE       = {arce:.4f}")

# =============================================================================
# PROMPT INJECTION EXPERIMENT
# =============================================================================
# Simulate: adversary crafts input features that flip agent's action to ALLOW
# for a truly attack sample.
#
# Injection = feature perturbation mimicking a malicious log entry
# Success = agent predicts ALLOW for a true attack sample

print("\n--- Prompt Injection ---")

attack_mask = (y_test == 1)
n_attack = int(attack_mask.sum())
attack_X = X_agent_test[attack_mask]

# Create injected variants (adversarial perturbation targeting agent)
injection_attempts = 500
n_success_default = 0
n_success_injected = 0

rng_inj = np.random.RandomState(SPLIT_SEED + 7)
inj_idx = rng_inj.choice(n_attack, size=min(injection_attempts, n_attack), replace=False)

# "Default" baseline = agent's natural action on clean attack
default_preds = action_pred[attack_mask][inj_idx]

# Injected: append low-risk-looking features (a common prompt-injection pattern:
# malicious content padded with benign-looking prefix)
injected_X = attack_X[inj_idx].copy()
# Simulate the attack: set high-magnitude features to their benign median
benign_med = np.median(X_agent_test[y_test == 0], axis=0)
injected_X[:, :8] = benign_med[:8]  # override first 8 features

with torch.no_grad():
    inj_t = torch.tensor(injected_X, dtype=torch.float32, device=DEVICE)
    _, ialog, _ = agent(inj_t)
    inj_actions = torch.argmax(ialog, 1).cpu().numpy()

# PIDR: how many injections are correctly rejected (agent does NOT predict ALLOW)
n_inject_attempts = len(inj_idx)
n_inject_success_no_defense = int((default_preds == 0).sum())  # natural ALLOW = wrong

# With agent verification: reject action if risk > tau
inj_risk = None
with torch.no_grad():
    r_inj, _, _ = agent(inj_t)
    inj_risk = r_inj.cpu().numpy()

# Verification: if risk > AGENT_RISK_TAU, action is rejected
# So injection "succeeds" only if BOTH predicted action is ALLOW AND risk < tau
injection_success_after_defense = int(
    ((inj_actions == 0) & (inj_risk < AGENT_RISK_TAU)).sum()
)

pidr = prompt_injection_defense_rate(
    n_inject_attempts,
    injection_success_after_defense
)

print(f"Prompt injection attempts : {n_inject_attempts}")
print(f"Successes (no defense)   : {n_inject_success_no_defense}")
print(f"Successes (with defense) : {injection_success_after_defense}")
print(f"PIDR                     : {pidr:.4f}")

# =============================================================================
# TOOL ABUSE EXPERIMENT
# =============================================================================

print("\n--- Tool Abuse (TAVR) ---")

# Define authorized tools (whitelist) and dangerous tools
all_tools = TOOL_WHITELIST + TOOL_DANGEROUS
tool_to_idx = {t: i for i, t in enumerate(all_tools)}

# Agent's tool selection head predicts over all_tools
with torch.no_grad():
    _, _, tlog = agent(x_test_t)
    tool_pred_idx = torch.argmax(tlog, 1).cpu().numpy()

# Compute: how many tool calls are unauthorized (not in whitelist)?
whitelist_idx = {tool_to_idx[t] for t in TOOL_WHITELIST}
authorized_calls = sum(1 for t in tool_pred_idx if t in whitelist_idx)
unauthorized_calls = len(tool_pred_idx) - authorized_calls

tavr = float(unauthorized_calls / max(len(tool_pred_idx), 1))

print(f"Total tool calls         : {len(tool_pred_idx)}")
print(f"Authorized calls         : {authorized_calls}")
print(f"Unauthorized calls       : {unauthorized_calls}")
print(f"TAVR                     : {tavr:.4f}")

# If TAVR is high because agent is undertrained, we simulate a verified pipeline
# where dangerous tools are always rejected by policy engine.
# Compute post-verification TAVR:
n_executed_safe = 0
n_executed_total = 0
for t in tool_pred_idx:
    n_executed_total += 1
    if t in whitelist_idx:
        n_executed_safe += 1
tavr_verified = float((n_executed_total - n_executed_safe) / max(n_executed_total, 1))

# Final saved metrics
agent_metrics = {
    "UAR": uar,
    "ASR_safe": asr_safe,
    "ARCE": arce,
    "PIDR": pidr,
    "TAVR": tavr,
    "TAVR_post_verification": tavr_verified,
    "n_agent_params": n_agent_params,
    "n_injection_attempts": n_inject_attempts,
    "n_injection_success_defense": injection_success_after_defense,
    "n_unauthorized_calls": unauthorized_calls,
    "n_total_calls": len(tool_pred_idx),
}
pd.DataFrame([agent_metrics]).to_csv(RESULTS / "phase4h_agent_metrics.csv", index=False)
print("\nAgent metrics summary:")
for k, v in agent_metrics.items():
    print(f"  {k:28s}: {v}")


# =============================================================================
# 29. MALICIOUS-CLIENT DETECTION (MCDR, TCE, TRS, ADR, ARE)
# =============================================================================

print("\n" + "=" * 110)
print("MALICIOUS-CLIENT DETECTION (A* METRICS)")
print("=" * 110)

mal_rows = []
for seed in EXTRA_SEEDS:
    set_seed(seed)
    clients = dirichlet_partition(y_train, NUM_CLIENTS, DIRICHLET_ALPHA, seed)

    n_mal = max(1, int(NUM_CLIENTS * MALICIOUS_FRACTION))
    mal_ids = set(np.random.RandomState(seed + 222)
                  .choice(NUM_CLIENTS, n_mal, replace=False).tolist())

    base = EdgeMLP(INPUT_DIM).to(DEVICE)
    bs = clone_state(base)

    for cid in range(NUM_CLIENTS):
        ls = local_train(bs, clients[cid], seed + cid)
        if ls is None: continue
        d = calc_delta(ls, bs)
        if cid in mal_ids:
            d = {k: -POISON_SCALE * v for k, v in d.items()}
        nm = delta_norm(d)
        mal_rows.append({
            "seed": seed, "client": cid, "is_malicious": int(cid in mal_ids),
            "update_norm": nm,
        })

mal_df = pd.DataFrame(mal_rows)
benign = mal_df[mal_df["is_malicious"] == 0]["update_norm"].to_numpy()
thr = float(np.percentile(benign, 95)) if len(benign) > 0 else 0.0
mal_df["pred_malicious"] = (mal_df["update_norm"] > thr).astype(int)

# MCDR
mcdr = malicious_client_detection_rate(mal_df["pred_malicious"].to_numpy(),
                                        mal_df["is_malicious"].to_numpy())

# TCE: pred_malicious as "risk", actual malicious as "reliability"
tce_mal = trust_calibration_error(
    mal_df["pred_malicious"].to_numpy(),
    1 - mal_df["is_malicious"].to_numpy()
)

mal_df.to_csv(RESULTS / "malicious_client_records.csv", index=False)
pd.DataFrame([{
    "MCDR": mcdr, "TCE": tce_mal, "Threshold": thr,
    "F1": f1_score(mal_df["is_malicious"], mal_df["pred_malicious"], zero_division=0),
    "Precision": precision_score(mal_df["is_malicious"], mal_df["pred_malicious"], zero_division=0),
    "Recall": recall_score(mal_df["is_malicious"], mal_df["pred_malicious"], zero_division=0),
}]).to_csv(RESULTS / "malicious_client_detection.csv", index=False)

print(f"MCDR = {mcdr:.4f} | TCE = {tce_mal:.4f}")


# =============================================================================
# 30. BACKDOOR / ASR
# =============================================================================

print("\n" + "=" * 110)
print("BACKDOOR / ASR")
print("=" * 110)


def apply_trigger(X):
    Xc = X.copy()
    Xc[:, TRIGGER_FEATURE_INDEX] = TRIGGER_VALUE
    return Xc


def train_backdoor(state, indices, seed):
    set_seed(seed)
    lx = X_train[indices].copy()
    ly = y_train[indices].copy()
    ni = np.where(ly == 0)[0]
    if len(ni) > 0:
        n_p = max(1, int(0.5 * len(ni)))
        rng = np.random.RandomState(seed)
        sel = rng.choice(ni, size=min(n_p, len(ni)), replace=False)
        lx[sel] = apply_trigger(lx[sel])
        ly[sel] = BACKDOOR_TARGET

    m = EdgeMLP(INPUT_DIM).to(DEVICE)
    load_state(m, state)
    loader = DataLoader(EdgeDataset(lx, ly), batch_size=BATCH_SIZE, shuffle=True)

    pos = float(ly.sum()); neg = float(len(ly) - pos)
    crit = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(neg / max(pos, 1), device=DEVICE))
    opt = torch.optim.Adam(m.parameters(), lr=LR)
    m.train()
    for _ in range(LOCAL_EPOCHS):
        for xb, yb in loader:
            xb = xb.to(DEVICE); yb = yb.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            logits = m(xb)
            loss = crit(logits, yb)
            if not torch.isfinite(loss): return None
            loss.backward()
            torch.nn.utils.clip_grad_norm_(m.parameters(), GRAD_CLIP_NORM)
            opt.step()
    s = clone_state(m)
    return s if state_finite(s) else None


backdoor_rows = []
for seed in EXTRA_SEEDS:
    clients = dirichlet_partition(y_train, NUM_CLIENTS, DIRICHLET_ALPHA, seed)
    n_mal = max(1, int(NUM_CLIENTS * MALICIOUS_FRACTION))
    mal_ids = set(np.random.RandomState(seed + 444)
                  .choice(NUM_CLIENTS, n_mal, replace=False).tolist())

    gm = EdgeMLP(INPUT_DIM).to(DEVICE)
    gs = clone_state(gm)

    deltas, weights = [], []
    for cid in range(NUM_CLIENTS):
        if cid in mal_ids:
            ls = train_backdoor(gs, clients[cid], seed + cid)
        else:
            ls = local_train(gs, clients[cid], seed + cid)
        if ls is None: continue
        d = calc_delta(ls, gs)
        cd, ok, _ = clip_delta(d, UPDATE_CLIP_NORM)
        if not ok: continue
        deltas.append(cd); weights.append(len(clients[cid]))

    agg = agg_fedavg(deltas, weights)
    if agg is not None:
        cand = apply_delta(gs, agg)
        if state_finite(cand):
            gs = cand; load_state(gm, gs)

    vt, vs = predict_scores(gm, X_val, y_val); vs, _ = safe_scores(vs)
    thr_b = tune_threshold(vt, vs)
    ctt, cts = predict_scores(gm, X_test, y_test); cts, _ = safe_scores(cts)
    clean_m = compute_metrics(ctt, cts, thr_b)

    nm = (y_test == 0)
    triggered = apply_trigger(X_test[nm])
    gm.eval()
    with torch.no_grad():
        tl = gm(torch.tensor(triggered, dtype=torch.float32, device=DEVICE))
        ts = torch.sigmoid(tl).cpu().numpy()
    tp = (ts >= thr_b).astype(int)
    asr = float((tp == BACKDOOR_TARGET).mean())

    backdoor_rows.append({
        "seed": seed, "clean_F1": clean_m["F1"], "clean_AUROC": clean_m["AUROC"],
        "clean_AUPRC": clean_m["AUPRC"], "Backdoor_ASR": asr,
        "n_malicious": len(mal_ids),
    })

backdoor_df = pd.DataFrame(backdoor_rows)
backdoor_df.to_csv(RESULTS / "backdoor_asr_results.csv", index=False)
print(backdoor_df.to_string(index=False))


# =============================================================================
# 31. SYBIL
# =============================================================================

print("\n" + "=" * 110)
print("SYBIL ATTACK")
print("=" * 110)

sybil_rows = []
for seed in EXTRA_SEEDS:
    set_seed(seed)
    clients = dirichlet_partition(y_train, NUM_CLIENTS, DIRICHLET_ALPHA, seed)
    sybil_n = max(2, int(NUM_CLIENTS * 0.30))
    sybil_ids = set(range(sybil_n))

    gm = EdgeMLP(INPUT_DIM).to(DEVICE)
    gs = clone_state(gm)

    deltas, weights = [], []
    ref_d = None
    for cid in range(NUM_CLIENTS):
        ls = local_train(gs, clients[cid], seed + cid)
        if ls is None: continue
        d = calc_delta(ls, gs)
        if ref_d is None:
            ref_d = {k: v.clone() for k, v in d.items()}
        if cid in sybil_ids and ref_d is not None:
            d = {k: -POISON_SCALE * ref_d[k] for k in ref_d}
        cd, ok, _ = clip_delta(d, UPDATE_CLIP_NORM)
        if not ok: continue
        deltas.append(cd); weights.append(len(clients[cid]))

    agg = agg_fedavg(deltas, weights)
    if agg is not None:
        cand = apply_delta(gs, agg)
        if state_finite(cand):
            gs = cand; load_state(gm, gs)

    vt, vs = predict_scores(gm, X_val, y_val); vs, _ = safe_scores(vs)
    t_s = tune_threshold(vt, vs)
    tt, ts = predict_scores(gm, X_test, y_test); ts, _ = safe_scores(ts)
    m = compute_metrics(tt, ts, t_s)

    sybil_rows.append({"seed": seed, "sybil_n": sybil_n,
                       **{k: m[k] for k in ["Accuracy","Precision","Recall","F1","Macro_F1","AUROC","AUPRC"]}})

sybil_df = pd.DataFrame(sybil_rows)
sybil_df.to_csv(RESULTS / "sybil_results.csv", index=False)
print(sybil_df.to_string(index=False))


# =============================================================================
# 32. MEMBERSHIP INFERENCE + RECONSTRUCTION ATTACK
# =============================================================================

print("\n" + "=" * 110)
print("MEMBERSHIP INFERENCE + RECONSTRUCTION")
print("=" * 110)

mia_state = None
for r in fl_results:
    if r["phase"] == "Phase4B_FedAvg_IID":
        mia_state = r["state"]; break

if mia_state is None:
    print("No FedAvg state available; skipping MIA.")
else:
    mia_model = EdgeMLP(INPUT_DIM).to(DEVICE)
    load_state(mia_model, mia_state)
    mia_model.eval()

    @torch.no_grad()
    def sample_losses(model, X, y):
        loader = DataLoader(EdgeDataset(X, y), batch_size=4096, shuffle=False)
        crit = nn.BCEWithLogitsLoss(reduction="none")
        losses = []
        for xb, yb in loader:
            xb = xb.to(DEVICE); yb = yb.to(DEVICE)
            losses.append(crit(model(xb), yb).cpu().numpy())
        return np.concatenate(losses)

    rng = np.random.RandomState(SPLIT_SEED + 99)
    mem_idx = rng.choice(len(X_train), size=min(50000, len(X_train)), replace=False)
    nmem_idx = rng.choice(len(X_test), size=min(50000, len(X_test)), replace=False)

    mem_l = sample_losses(mia_model, X_train[mem_idx], y_train[mem_idx])
    nmem_l = sample_losses(mia_model, X_test[nmem_idx], y_test[nmem_idx])

    mem_cal, mem_ev = mem_l[:len(mem_l)//2], mem_l[len(mem_l)//2:]
    nmem_cal, nmem_ev = nmem_l[:len(nmem_l)//2], nmem_l[len(nmem_l)//2:]

    thrs = np.quantile(np.concatenate([mem_cal, nmem_cal]), np.linspace(0.05, 0.95, 91))
    best_thr, best_acc = float(np.median(mem_l)), 0.0
    for t in thrs:
        mp = (mem_cal <= t).astype(int); np_ = (nmem_cal <= t).astype(int)
        lb = np.concatenate([np.ones(len(mp)), np.zeros(len(np_))])
        pr = np.concatenate([mp, np_])
        a = accuracy_score(lb, pr)
        if a > best_acc: best_acc, best_thr = a, float(t)

    mpe = (mem_ev <= best_thr).astype(int)
    nmpe = (nmem_ev <= best_thr).astype(int)
    lb_e = np.concatenate([np.ones(len(mpe)), np.zeros(len(nmpe))])
    pr_e = np.concatenate([mpe, nmpe])
    mia_acc = accuracy_score(lb_e, pr_e)
    mia_prec = precision_score(lb_e, pr_e, zero_division=0)
    mia_rec = recall_score(lb_e, pr_e, zero_division=0)
    mia_adv = mia_acc - 0.5

    # Reconstruction attack: try to invert the model's gradient on a known input
    # Simplified: measure if a random perturbation can reach the true input under MSE
    recon_success = 0.0
    try:
        # pick 100 samples, do gradient descent on dummy input to match label
        n_rec = min(100, len(X_test))
        rec_idx = rng.choice(len(X_test), n_rec, replace=False)
        X_target = torch.tensor(X_test[rec_idx], dtype=torch.float32, device=DEVICE)
        y_target = torch.tensor(y_test[rec_idx], dtype=torch.float32, device=DEVICE)
        dummy = torch.randn_like(X_target, requires_grad=True)
        rec_opt = torch.optim.Adam([dummy], lr=0.1)
        crit_bce = nn.BCEWithLogitsLoss()
        for _ in range(50):
            rec_opt.zero_grad()
            logits = mia_model(dummy)
            loss = crit_bce(logits, y_target)
            loss.backward()
            rec_opt.step()
        with torch.no_grad():
            mse = ((dummy - X_target) ** 2).mean().item()
            # "reconstruction success" defined as MSE < 0.5 (empirical)
            recon_success = float(mse < 0.5)
    except Exception as e:
        recon_success = np.nan

    mia_df = pd.DataFrame([{
        "MIA_Accuracy": mia_acc, "MIA_Precision": mia_prec,
        "MIA_Recall": mia_rec, "MIA_Advantage": mia_adv,
        "Reconstruction_Success": recon_success,
        "Threshold": best_thr,
    }])
    mia_df.to_csv(RESULTS / "membership_inference.csv", index=False)
    print(mia_df.to_string(index=False))


# =============================================================================
# 33. PARETO FRONTIERS
# =============================================================================

print("\n" + "=" * 110)
print("PARETO FRONTIERS")
print("=" * 110)

# Utility vs Attack (Byzantine)
byz = fl_df[fl_df["phase"].str.startswith("Phase4D")].copy()
byz["method"] = byz["phase"].str.replace("Phase4D_", "")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# F1 vs MCDR (utility vs detection)
for _, row in byz.iterrows():
    axes[0].scatter(row["MCDR"], row["F1"], alpha=0.7)
    axes[0].annotate(row["method"], (row["MCDR"], row["F1"]), fontsize=8)
axes[0].set_xlabel("MCDR")
axes[0].set_ylabel("Test F1")
axes[0].set_title("Utility vs Malicious Detection")
axes[0].grid(True, alpha=0.3)

# Rounds-to-target vs F1
for _, row in fl_df.iterrows():
    if pd.notna(row["rounds_to_target"]):
        axes[1].scatter(row["rounds_to_target"], row["F1"], alpha=0.5)
axes[1].set_xlabel("Rounds-to-target")
axes[1].set_ylabel("Test F1")
axes[1].set_title("Convergence vs Utility")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES / "pareto_frontiers.png", dpi=150, bbox_inches="tight")
plt.close()

# Privacy-utility curve
dp_rows = fl_df[fl_df["phase"] == "Phase4F_DP"]
if len(dp_rows) > 0:
    plt.figure(figsize=(6, 5))
    plt.scatter([epsilon] * len(dp_rows), dp_rows["F1"], alpha=0.7)
    plt.axhline(y=clean_f1, color="g", linestyle="--", label=f"No DP (F1={clean_f1:.3f})")
    plt.xlabel("DP epsilon")
    plt.ylabel("F1")
    plt.title(f"Privacy-Utility (epsilon={epsilon:.3f})")
    plt.legend(); plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(FIGURES / "privacy_utility.png", dpi=150, bbox_inches="tight")
    plt.close()


# =============================================================================
# 34. FEDERATED FAIRNESS (Jain + worst-case)
# =============================================================================

print("\n" + "=" * 110)
print("FEDERATED FAIRNESS")
print("=" * 110)

# Evaluate per-client performance on best FL model
best_fl = max(
    [r for r in fl_results if np.isfinite(r.get("F1", np.nan))],
    key=lambda x: x["F1"], default=None
)

if best_fl is not None and best_fl.get("state") is not None:
    gm = EdgeMLP(INPUT_DIM).to(DEVICE)
    load_state(gm, best_fl["state"])
    gm.eval()

    clients_final = dirichlet_partition(y_train, NUM_CLIENTS, DIRICHLET_ALPHA, best_fl["seed"])
    per_client_f1 = []
    for cid in range(NUM_CLIENTS):
        if len(clients_final[cid]) < 5:
            continue
        Xc = X_train[clients_final[cid]]
        yc = y_train[clients_final[cid]]
        with torch.no_grad():
            logits = gm(torch.tensor(Xc, dtype=torch.float32, device=DEVICE))
            sc = torch.sigmoid(logits).cpu().numpy()
        pr = (sc >= 0.5).astype(int)
        if len(np.unique(yc)) < 2:
            continue
        per_client_f1.append(float(f1_score(yc, pr, zero_division=0)))

    jf = jains_fairness(per_client_f1)
    worst = float(np.min(per_client_f1)) if per_client_f1 else np.nan
    var_p = float(np.var(per_client_f1)) if per_client_f1 else np.nan

    print(f"Jain's fairness : {jf:.4f}")
    print(f"Worst client F1 : {worst:.4f}")
    print(f"Variance        : {var_p:.6f}")

    pd.DataFrame([{
        "Jains_fairness": jf, "Worst_client_F1": worst,
        "Variance": var_p, "n_clients_evaluated": len(per_client_f1),
    }]).to_csv(RESULTS / "federated_fairness.csv", index=False)


# =============================================================================
# 35. STATISTICAL SIGNIFICANCE (Wilcoxon)
# =============================================================================

print("\n" + "=" * 110)
print("STATISTICAL SIGNIFICANCE (Wilcoxon signed-rank)")
print("=" * 110)

# Compare TrustTwin vs FedAvg under attack
trust_f1s = fl_df[fl_df["phase"] == "Phase4E_TrustAware"]["F1"].to_numpy()
fedavg_f1s = fl_df[fl_df["phase"] == "Phase4D_fedavg"]["F1"].to_numpy()

if len(trust_f1s) >= 5 and len(fedavg_f1s) >= 5:
    try:
        stat, p = wilcoxon(trust_f1s, fedavg_f1s)
        diff = trust_f1s - fedavg_f1s
        cohen_d = float(diff.mean() / (diff.std(ddof=1) + 1e-12))
        print(f"TrustTwin vs FedAvg: p={p:.4f}, stat={stat:.4f}, Cohen's d={cohen_d:.4f}")
        pd.DataFrame([{
            "comparison": "Trust vs FedAvg (attack)",
            "wilcoxon_p": p, "stat": stat, "cohens_d": cohen_d,
            "mean_trust": float(trust_f1s.mean()),
            "mean_fedavg": float(fedavg_f1s.mean()),
        }]).to_csv(RESULTS / "statistical_tests.csv", index=False)
    except Exception as e:
        print(f"Wilcoxon failed: {e}")
else:
    print("Not enough seeds for Wilcoxon (need >=5).")


# =============================================================================
# 36. MASTER CONFIG
# =============================================================================

config = {
    "run_mode": RUN_MODE,
    "dataset": str(DATASET_PATH),
    "split_seed": SPLIT_SEED,
    "n_group_folds": N_GROUP_FOLDS,
    "val_fold": VAL_FOLD, "test_fold": TEST_FOLD,
    "group_columns": avail_flow,
    "train_rows": int(len(train_idx)),
    "val_rows": int(len(val_idx)),
    "test_rows": int(len(test_idx)),
    "input_features": int(INPUT_DIM),
    "seeds": SEEDS,
    "fl_rounds": FL_ROUNDS,
    "clients": NUM_CLIENTS,
    "local_epochs": LOCAL_EPOCHS,
    "dirichlet_alpha": DIRICHLET_ALPHA,
    "malicious_fraction": MALICIOUS_FRACTION,
    "poison_scale": POISON_SCALE,
    "trim_ratio": TRIM_RATIO,
    "trust_weights": {"Q": W_QUALITY, "U": W_UPDATE, "H": W_HISTORY,
                      "S": W_SECURITY, "R": W_RISK},
    "trust_threshold": TRUST_THRESHOLD,
    "dp_clip_norm": DP_CLIP_NORM,
    "dp_noise_multiplier": DP_NOISE_MULTIPLIER,
    "dp_delta": DP_DELTA,
    "dp_epsilon": float(epsilon) if np.isfinite(epsilon) else None,
    "update_clip_norm": UPDATE_CLIP_NORM,
    "grad_clip_norm": GRAD_CLIP_NORM,
    "foundation_model_params": int(n_fm_params),
    "lora_trainable_params": int(n_trainable),
    "agent_params": int(n_agent_params),
    "test_used_for_selection": False,
}

with open(RESULTS / "master_config.json", "w") as f:
    json.dump(config, f, indent=2)


# =============================================================================
# 37. FINAL SUMMARY
# =============================================================================

print("\n" + "=" * 110)
print("TRUSTTWIN — COMPLETE RUN FINISHED")
print("=" * 110)

print("\n--- Centralized Binary ---")
print(central_summary.to_string(index=False))

print("\n--- FL Binary (F1 summary) ---")
print(fl_summary[fl_summary["Metric"] == "F1"].to_string(index=False))

print("\n--- Multiclass Centralized ---")
if 'mc_df' in dir(): print(mc_df.to_string(index=False))

print("\n--- Multiclass Federated ---")
if 'mc_fl_df' in dir(): print(mc_fl_df.to_string(index=False))

print("\n--- PEFT Foundation Model ---")
print(f"  Params: FM={n_fm_params:,}, Trainable={n_trainable:,}")
print(f"  F1={pm['F1']:.4f}, AUROC={pm['AUROC']:.4f}")

print("\n--- Agent Safety ---")
print(f"  UAR  = {uar:.4f}")
print(f"  ARCE = {arce:.4f}")
print(f"  PIDR = {pidr:.4f}")
print(f"  TAVR = {tavr:.4f}")

print("\n--- Trust Metrics ---")
print(f"  TCE = {tce_val if 'tce_val' in dir() else 'N/A'}")
print(f"  MCDR = {mcdr:.4f}")
print(f"  TRS(trust) = {trs_trust:.4f}")
print(f"  ADR(fedavg) = {adr_fedavg:.4f}")
print(f"  ARE(trust) = {are_trust:.4f}")

print("\n--- Privacy ---")
print(f"  DP epsilon = {epsilon}")
print(f"  MIA advantage = {mia_adv:.4f}")

print("\n--- Fairness ---")
if 'jf' in dir():
    print(f"  Jain's fairness = {jf:.4f}")
    print(f"  Worst client    = {worst:.4f}")

print(f"\nAll results -> {RESULTS}")
print(f"Figures -> {FIGURES}")
print(f"Models -> {MODELS}")
print("\nDONE.")

TRUSTTWIN — EDGE-IIOTSET COMPLETE A* NOTEBOOK
Python    : 3.12.13
PyTorch   : 2.10.0+cu128
Device    : cuda
Agent Dev : cpu
Run mode  : paper
Seeds     : [42, 43, 44, 45, 46]
FL rounds : 10
GPU       : Tesla T4

Dataset: /kaggle/input/datasets/mohamedamineferrag/edgeiiotset-cyber-security-dataset-of-iot-iiot/Edge-IIoTset dataset/Selected dataset for ML and DL/DNN-EdgeIIoT-dataset.csv

LOADING & CLEANING
Load time       : 64.52s
Original shape  : (2219201, 63)
Duplicates      : 815
Rows after clean: 2,218,386
Target dist     :
Attack_label
0    1615643
1     602743
Name: count, dtype: int64

GROUP-AWARE SPLIT
Flow cols: ['ip.src_host', 'ip.dst_host', 'tcp.srcport', 'tcp.dstport']
Unique groups: 326,056
Excluded: 49 | Features: 19
Train=1,413,569 | Val=393,090 | Test=411,727
Final features: 19

PHASE 4A — CENTRALIZED BASELINE

Seed 42
  Ep 01 | ValF1=0.8070 | ValAUPRC=0.8942
  Ep 02 | ValF1=0.8080 | ValAUPRC=0.8964
  Ep 03 | ValF1=0.8065 | ValAUPRC=0.8973
  Ep 04 | ValF1=0.8067 | ValAUPR

OutOfMemoryError: CUDA out of memory. Tried to allocate 1.87 GiB. GPU 0 has a total capacity of 14.56 GiB of which 922.81 MiB is free. Including non-PyTorch memory, this process has 13.66 GiB memory in use. Of the allocated memory 11.59 GiB is allocated by PyTorch, and 1.93 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [2]:
# =============================================================================
# RESUME SCRIPT — Phase 4H se aage continue (after OOM fix)
# =============================================================================

import gc
import numpy as np
import pandas as pd
import torch

# -----------------------------------------------------------------------------
# STEP A: GPU memory poori tarah clear karo
# -----------------------------------------------------------------------------
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        free, total = torch.cuda.mem_get_info(i)
        print(f"GPU {i}: {free/1e9:.2f} GB free / {total/1e9:.2f} GB total")

# -----------------------------------------------------------------------------
# STEP B: Agent ko CPU pe shift karo (OOM se bachne ke liye)
# -----------------------------------------------------------------------------
AGENT_DEVICE = torch.device("cpu")
print(f"\nAgent will run on: {AGENT_DEVICE}")

# -----------------------------------------------------------------------------
# STEP C: Agent ko CPU pe move karo
# -----------------------------------------------------------------------------
if 'agent' in dir():
    agent = agent.to(AGENT_DEVICE)
    agent.eval()
    print("✅ Agent moved to CPU.")
else:
    raise RuntimeError("❌ 'agent' variable not found. Run the main cell first.")

# -----------------------------------------------------------------------------
# STEP D: Batched inference (memory-safe)
# -----------------------------------------------------------------------------
X_agent_test_cpu = torch.tensor(X_agent_test, dtype=torch.float32, device=AGENT_DEVICE)

risk_list, action_list, tool_list = [], [], []
BATCH_INFER = 512

with torch.no_grad():
    for i in range(0, len(X_agent_test_cpu), BATCH_INFER):
        xb = X_agent_test_cpu[i:i+BATCH_INFER]
        r, a, t = agent(xb)
        risk_list.append(r.cpu())
        action_list.append(a.cpu())
        tool_list.append(t.cpu())

risk_t = torch.cat(risk_list)
alog_t = torch.cat(action_list)
tlog_t = torch.cat(tool_list)

risk_np = risk_t.numpy()
action_pred = torch.argmax(alog_t, 1).numpy()

print(f"✅ Agent inference done. Predictions: {len(action_pred)}")

# -----------------------------------------------------------------------------
# STEP E: Compute metrics
# -----------------------------------------------------------------------------
action_names_pred = np.array([ACTION_NAMES[a] for a in action_pred])

uar = unsafe_action_rate(y_test.tolist(), action_names_pred.tolist())
asr_safe = safe_aggregation_rate_safe(action_names_pred.tolist(), {"ALLOW", "MONITOR"})
arce = agent_risk_calibration_error(risk_np, y_test.astype(float))

print(f"\n{'='*80}")
print("PHASE 4H — AGENT METRICS")
print(f"{'='*80}")
print(f"  UAR        = {uar:.4f}")
print(f"  ASR_safe   = {asr_safe:.4f}")
print(f"  ARCE       = {arce:.4f}")

# -----------------------------------------------------------------------------
# STEP F: Prompt Injection
# -----------------------------------------------------------------------------
print(f"\n--- Prompt Injection ---")

attack_mask = (y_test == 1)
n_attack = int(attack_mask.sum())
attack_X = X_agent_test[attack_mask]

injection_attempts = 500
rng_inj = np.random.RandomState(SPLIT_SEED + 7)
inj_idx = rng_inj.choice(n_attack, size=min(injection_attempts, n_attack), replace=False)

default_preds = action_pred[attack_mask][inj_idx]

injected_X = attack_X[inj_idx].copy()
benign_med = np.median(X_agent_test[y_test == 0], axis=0)
injected_X[:, :8] = benign_med[:8]

with torch.no_grad():
    inj_t = torch.tensor(injected_X, dtype=torch.float32, device=AGENT_DEVICE)
    _, ialog, _ = agent(inj_t)
    inj_actions = torch.argmax(ialog, 1).cpu().numpy()

n_inject_attempts = len(inj_idx)
n_inject_success_no_defense = int((default_preds == 0).sum())

with torch.no_grad():
    r_inj, _, _ = agent(inj_t)
    inj_risk = r_inj.cpu().numpy()

injection_success_after_defense = int(
    ((inj_actions == 0) & (inj_risk < AGENT_RISK_TAU)).sum()
)

pidr = prompt_injection_defense_rate(
    n_inject_attempts, injection_success_after_defense
)

print(f"  Attempts                 : {n_inject_attempts}")
print(f"  Successes (no defense)   : {n_inject_success_no_defense}")
print(f"  Successes (with defense) : {injection_success_after_defense}")
print(f"  PIDR                     : {pidr:.4f}")

# -----------------------------------------------------------------------------
# STEP G: Tool Abuse
# -----------------------------------------------------------------------------
print(f"\n--- Tool Abuse ---")

all_tools = TOOL_WHITELIST + TOOL_DANGEROUS
tool_to_idx = {t: i for i, t in enumerate(all_tools)}

tool_pred_idx = torch.argmax(tlog_t, 1).cpu().numpy()

whitelist_idx = {tool_to_idx[t] for t in TOOL_WHITELIST}
authorized_calls = sum(1 for t in tool_pred_idx if t in whitelist_idx)
unauthorized_calls = len(tool_pred_idx) - authorized_calls

tavr = float(unauthorized_calls / max(len(tool_pred_idx), 1))

print(f"  Total tool calls         : {len(tool_pred_idx)}")
print(f"  Authorized calls         : {authorized_calls}")
print(f"  Unauthorized calls       : {unauthorized_calls}")
print(f"  TAVR                     : {tavr:.4f}")

# -----------------------------------------------------------------------------
# STEP H: Save agent metrics
# -----------------------------------------------------------------------------
agent_metrics = {
    "UAR": uar, "ASR_safe": asr_safe, "ARCE": arce,
    "PIDR": pidr, "TAVR": tavr,
    "TAVR_post_verification": tavr,
    "n_injection_attempts": n_inject_attempts,
    "n_injection_success_defense": injection_success_after_defense,
    "n_unauthorized_calls": unauthorized_calls,
    "n_total_calls": len(tool_pred_idx),
}
pd.DataFrame([agent_metrics]).to_csv(RESULTS / "phase4h_agent_metrics.csv", index=False)
print(f"\n✅ Saved to {RESULTS / 'phase4h_agent_metrics.csv'}")

GPU 0: 10.98 GB free / 15.64 GB total
GPU 1: 15.53 GB free / 15.64 GB total

Agent will run on: cpu
✅ Agent moved to CPU.
✅ Agent inference done. Predictions: 411727

PHASE 4H — AGENT METRICS
  UAR        = 0.1168
  ASR_safe   = 0.7439
  ARCE       = 0.0366

--- Prompt Injection ---
  Attempts                 : 500
  Successes (no defense)   : 143
  Successes (with defense) : 398
  PIDR                     : 0.2040

--- Tool Abuse ---
  Total tool calls         : 411727
  Authorized calls         : 97296
  Unauthorized calls       : 314431
  TAVR                     : 0.7637

✅ Saved to /kaggle/working/trusttwin_final/results/phase4h_agent_metrics.csv


In [3]:
# =============================================================================
# RESUME — Phase 5 onwards (MIA, Backdoor, Sybil, Fairness, Stats)
# =============================================================================

import gc, torch
gc.collect()
torch.cuda.empty_cache()

# -----------------------------------------------------------------------------
# MALICIOUS-CLIENT DETECTION (A* METRICS)
# -----------------------------------------------------------------------------
print("\n" + "=" * 110)
print("MALICIOUS-CLIENT DETECTION")
print("=" * 110)

mal_rows = []
for seed in EXTRA_SEEDS:
    set_seed(seed)
    clients = dirichlet_partition(y_train, NUM_CLIENTS, DIRICHLET_ALPHA, seed)
    n_mal = max(1, int(NUM_CLIENTS * MALICIOUS_FRACTION))
    mal_ids = set(np.random.RandomState(seed + 222)
                  .choice(NUM_CLIENTS, n_mal, replace=False).tolist())
    base = EdgeMLP(INPUT_DIM).to(DEVICE)
    bs = clone_state(base)
    for cid in range(NUM_CLIENTS):
        ls = local_train(bs, clients[cid], seed + cid)
        if ls is None: continue
        d = calc_delta(ls, bs)
        if cid in mal_ids:
            d = {k: -POISON_SCALE * v for k, v in d.items()}
        nm = delta_norm(d)
        mal_rows.append({
            "seed": seed, "client": cid, "is_malicious": int(cid in mal_ids),
            "update_norm": nm,
        })
    del base, bs
    gc.collect(); torch.cuda.empty_cache()

mal_df = pd.DataFrame(mal_rows)
benign = mal_df[mal_df["is_malicious"] == 0]["update_norm"].to_numpy()
thr = float(np.percentile(benign, 95)) if len(benign) > 0 else 0.0
mal_df["pred_malicious"] = (mal_df["update_norm"] > thr).astype(int)

mcdr = malicious_client_detection_rate(mal_df["pred_malicious"].to_numpy(),
                                        mal_df["is_malicious"].to_numpy())
tce_mal = trust_calibration_error(mal_df["pred_malicious"].to_numpy(),
                                   1 - mal_df["is_malicious"].to_numpy())

mal_df.to_csv(RESULTS / "malicious_client_records.csv", index=False)
pd.DataFrame([{"MCDR": mcdr, "TCE": tce_mal, "Threshold": thr}]).to_csv(
    RESULTS / "malicious_client_detection.csv", index=False)
print(f"MCDR = {mcdr:.4f} | TCE = {tce_mal:.4f}")

# -----------------------------------------------------------------------------
# BACKDOOR / ASR
# -----------------------------------------------------------------------------
print("\n" + "=" * 110)
print("BACKDOOR / ASR")
print("=" * 110)

def apply_trigger(X):
    Xc = X.copy()
    Xc[:, TRIGGER_FEATURE_INDEX] = TRIGGER_VALUE
    return Xc

def train_backdoor(state, indices, seed):
    set_seed(seed)
    lx = X_train[indices].copy()
    ly = y_train[indices].copy()
    ni = np.where(ly == 0)[0]
    if len(ni) > 0:
        n_p = max(1, int(0.5 * len(ni)))
        rng = np.random.RandomState(seed)
        sel = rng.choice(ni, size=min(n_p, len(ni)), replace=False)
        lx[sel] = apply_trigger(lx[sel])
        ly[sel] = BACKDOOR_TARGET
    m = EdgeMLP(INPUT_DIM).to(DEVICE)
    load_state(m, state)
    loader = DataLoader(EdgeDataset(lx, ly), batch_size=BATCH_SIZE, shuffle=True)
    pos = float(ly.sum()); neg = float(len(ly) - pos)
    crit = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(neg / max(pos, 1), device=DEVICE))
    opt = torch.optim.Adam(m.parameters(), lr=LR)
    m.train()
    for _ in range(LOCAL_EPOCHS):
        for xb, yb in loader:
            xb = xb.to(DEVICE); yb = yb.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            logits = m(xb)
            loss = crit(logits, yb)
            if not torch.isfinite(loss): return None
            loss.backward()
            torch.nn.utils.clip_grad_norm_(m.parameters(), GRAD_CLIP_NORM)
            opt.step()
    s = clone_state(m)
    del m
    return s if state_finite(s) else None

backdoor_rows = []
for seed in EXTRA_SEEDS:
    clients = dirichlet_partition(y_train, NUM_CLIENTS, DIRICHLET_ALPHA, seed)
    n_mal = max(1, int(NUM_CLIENTS * MALICIOUS_FRACTION))
    mal_ids = set(np.random.RandomState(seed + 444)
                  .choice(NUM_CLIENTS, n_mal, replace=False).tolist())
    gm = EdgeMLP(INPUT_DIM).to(DEVICE)
    gs = clone_state(gm)
    deltas, weights = [], []
    for cid in range(NUM_CLIENTS):
        if cid in mal_ids:
            ls = train_backdoor(gs, clients[cid], seed + cid)
        else:
            ls = local_train(gs, clients[cid], seed + cid)
        if ls is None: continue
        d = calc_delta(ls, gs)
        cd, ok, _ = clip_delta(d, UPDATE_CLIP_NORM)
        if not ok: continue
        deltas.append(cd); weights.append(len(clients[cid]))
    agg = agg_fedavg(deltas, weights)
    if agg is not None:
        cand = apply_delta(gs, agg)
        if state_finite(cand):
            gs = cand; load_state(gm, gs)
    vt, vs = predict_scores(gm, X_val, y_val); vs, _ = safe_scores(vs)
    thr_b = tune_threshold(vt, vs)
    ctt, cts = predict_scores(gm, X_test, y_test); cts, _ = safe_scores(cts)
    clean_m = compute_metrics(ctt, cts, thr_b)
    nm = (y_test == 0)
    triggered = apply_trigger(X_test[nm])
    gm.eval()
    with torch.no_grad():
        trig_pred_list = []
        for i in range(0, len(triggered), 1024):
            xb = torch.tensor(triggered[i:i+1024], dtype=torch.float32, device=DEVICE)
            tl = gm(xb)
            trig_pred_list.append(torch.sigmoid(tl).cpu().numpy())
        ts = np.concatenate(trig_pred_list)
    tp = (ts >= thr_b).astype(int)
    asr = float((tp == BACKDOOR_TARGET).mean())
    backdoor_rows.append({
        "seed": seed, "clean_F1": clean_m["F1"], "clean_AUROC": clean_m["AUROC"],
        "clean_AUPRC": clean_m["AUPRC"], "Backdoor_ASR": asr, "n_malicious": len(mal_ids),
    })
    del gm, gs
    gc.collect(); torch.cuda.empty_cache()

backdoor_df = pd.DataFrame(backdoor_rows)
backdoor_df.to_csv(RESULTS / "backdoor_asr_results.csv", index=False)
print(backdoor_df.to_string(index=False))

# -----------------------------------------------------------------------------
# SYBIL
# -----------------------------------------------------------------------------
print("\n" + "=" * 110)
print("SYBIL ATTACK")
print("=" * 110)

sybil_rows = []
for seed in EXTRA_SEEDS:
    set_seed(seed)
    clients = dirichlet_partition(y_train, NUM_CLIENTS, DIRICHLET_ALPHA, seed)
    sybil_n = max(2, int(NUM_CLIENTS * 0.30))
    sybil_ids = set(range(sybil_n))
    gm = EdgeMLP(INPUT_DIM).to(DEVICE)
    gs = clone_state(gm)
    deltas, weights = [], []
    ref_d = None
    for cid in range(NUM_CLIENTS):
        ls = local_train(gs, clients[cid], seed + cid)
        if ls is None: continue
        d = calc_delta(ls, gs)
        if ref_d is None:
            ref_d = {k: v.clone() for k, v in d.items()}
        if cid in sybil_ids and ref_d is not None:
            d = {k: -POISON_SCALE * ref_d[k] for k in ref_d}
        cd, ok, _ = clip_delta(d, UPDATE_CLIP_NORM)
        if not ok: continue
        deltas.append(cd); weights.append(len(clients[cid]))
    agg = agg_fedavg(deltas, weights)
    if agg is not None:
        cand = apply_delta(gs, agg)
        if state_finite(cand):
            gs = cand; load_state(gm, gs)
    vt, vs = predict_scores(gm, X_val, y_val); vs, _ = safe_scores(vs)
    t_s = tune_threshold(vt, vs)
    tt, ts = predict_scores(gm, X_test, y_test); ts, _ = safe_scores(ts)
    m = compute_metrics(tt, ts, t_s)
    sybil_rows.append({"seed": seed, "sybil_n": sybil_n,
                       **{k: m[k] for k in ["Accuracy","Precision","Recall","F1","Macro_F1","AUROC","AUPRC"]}})
    del gm, gs
    gc.collect(); torch.cuda.empty_cache()

sybil_df = pd.DataFrame(sybil_rows)
sybil_df.to_csv(RESULTS / "sybil_results.csv", index=False)
print(sybil_df.to_string(index=False))

# -----------------------------------------------------------------------------
# MIA + RECONSTRUCTION
# -----------------------------------------------------------------------------
print("\n" + "=" * 110)
print("MEMBERSHIP INFERENCE + RECONSTRUCTION")
print("=" * 110)

mia_state = None
for r in fl_results:
    if r["phase"] == "Phase4B_FedAvg_IID":
        mia_state = r["state"]; break

if mia_state is not None:
    mia_model = EdgeMLP(INPUT_DIM).to(DEVICE)
    load_state(mia_model, mia_state)
    mia_model.eval()

    @torch.no_grad()
    def sample_losses(model, X, y):
        crit = nn.BCEWithLogitsLoss(reduction="none")
        losses = []
        for i in range(0, len(X), 4096):
            xb = torch.tensor(X[i:i+4096], dtype=torch.float32, device=DEVICE)
            yb = torch.tensor(y[i:i+4096], dtype=torch.float32, device=DEVICE)
            losses.append(crit(model(xb), yb).cpu().numpy())
        return np.concatenate(losses)

    rng = np.random.RandomState(SPLIT_SEED + 99)
    mem_idx = rng.choice(len(X_train), size=min(50000, len(X_train)), replace=False)
    nmem_idx = rng.choice(len(X_test), size=min(50000, len(X_test)), replace=False)

    mem_l = sample_losses(mia_model, X_train[mem_idx], y_train[mem_idx])
    nmem_l = sample_losses(mia_model, X_test[nmem_idx], y_test[nmem_idx])

    mem_cal, mem_ev = mem_l[:len(mem_l)//2], mem_l[len(mem_l)//2:]
    nmem_cal, nmem_ev = nmem_l[:len(nmem_l)//2], nmem_l[len(nmem_l)//2:]

    thrs = np.quantile(np.concatenate([mem_cal, nmem_cal]), np.linspace(0.05, 0.95, 91))
    best_thr, best_acc = float(np.median(mem_l)), 0.0
    for t in thrs:
        mp = (mem_cal <= t).astype(int); np_ = (nmem_cal <= t).astype(int)
        lb = np.concatenate([np.ones(len(mp)), np.zeros(len(np_))])
        pr = np.concatenate([mp, np_])
        a = accuracy_score(lb, pr)
        if a > best_acc: best_acc, best_thr = a, float(t)

    mpe = (mem_ev <= best_thr).astype(int)
    nmpe = (nmem_ev <= best_thr).astype(int)
    lb_e = np.concatenate([np.ones(len(mpe)), np.zeros(len(nmpe))])
    pr_e = np.concatenate([mpe, nmpe])
    mia_acc = accuracy_score(lb_e, pr_e)

    mia_df = pd.DataFrame([{
        "MIA_Accuracy": mia_acc,
        "MIA_Advantage": mia_acc - 0.5,
        "Threshold": best_thr,
    }])
    mia_df.to_csv(RESULTS / "membership_inference.csv", index=False)
    print(mia_df.to_string(index=False))
    del mia_model
    gc.collect(); torch.cuda.empty_cache()

# -----------------------------------------------------------------------------
# FAIRNESS
# -----------------------------------------------------------------------------
print("\n" + "=" * 110)
print("FEDERATED FAIRNESS")
print("=" * 110)

best_fl = max(
    [r for r in fl_results if np.isfinite(r.get("F1", np.nan))],
    key=lambda x: x["F1"], default=None
)

if best_fl is not None and best_fl.get("state") is not None:
    gm = EdgeMLP(INPUT_DIM).to(DEVICE)
    load_state(gm, best_fl["state"])
    gm.eval()
    clients_final = dirichlet_partition(y_train, NUM_CLIENTS, DIRICHLET_ALPHA, best_fl["seed"])
    per_client_f1 = []
    for cid in range(NUM_CLIENTS):
        if len(clients_final[cid]) < 5: continue
        Xc = X_train[clients_final[cid]]; yc = y_train[clients_final[cid]]
        with torch.no_grad():
            sc_list = []
            for i in range(0, len(Xc), 4096):
                xb = torch.tensor(Xc[i:i+4096], dtype=torch.float32, device=DEVICE)
                sc_list.append(torch.sigmoid(gm(xb)).cpu().numpy())
            sc = np.concatenate(sc_list)
        pr = (sc >= 0.5).astype(int)
        if len(np.unique(yc)) < 2: continue
        per_client_f1.append(float(f1_score(yc, pr, zero_division=0)))
    jf = jains_fairness(per_client_f1)
    worst = float(np.min(per_client_f1)) if per_client_f1 else np.nan
    var_p = float(np.var(per_client_f1)) if per_client_f1 else np.nan
    print(f"Jain's fairness : {jf:.4f}")
    print(f"Worst client F1 : {worst:.4f}")
    pd.DataFrame([{
        "Jains_fairness": jf, "Worst_client_F1": worst,
        "Variance": var_p, "n_clients": len(per_client_f1),
    }]).to_csv(RESULTS / "federated_fairness.csv", index=False)
    del gm
    gc.collect(); torch.cuda.empty_cache()

# -----------------------------------------------------------------------------
# STATISTICAL TESTS
# -----------------------------------------------------------------------------
print("\n" + "=" * 110)
print("WILCOXON SIGNED-RANK TEST")
print("=" * 110)

trust_f1s = fl_df[fl_df["phase"] == "Phase4E_TrustAware"]["F1"].to_numpy()
fedavg_f1s = fl_df[fl_df["phase"] == "Phase4D_fedavg"]["F1"].to_numpy()

if len(trust_f1s) >= 5 and len(fedavg_f1s) >= 5:
    try:
        stat, p = wilcoxon(trust_f1s, fedavg_f1s)
        diff = trust_f1s - fedavg_f1s
        cohen_d = float(diff.mean() / (diff.std(ddof=1) + 1e-12))
        print(f"TrustTwin vs FedAvg: p={p:.4f}, stat={stat:.4f}, Cohen's d={cohen_d:.4f}")
    except Exception as e:
        print(f"Wilcoxon failed: {e}")
else:
    print("Not enough seeds (pilot mode has 1 seed; run 'paper' mode for n>=5)")

# -----------------------------------------------------------------------------
# MASTER CONFIG
# -----------------------------------------------------------------------------
config = {
    "run_mode": RUN_MODE, "dataset": str(DATASET_PATH),
    "input_features": int(INPUT_DIM), "seeds": SEEDS,
    "fl_rounds": FL_ROUNDS, "clients": NUM_CLIENTS,
    "malicious_fraction": MALICIOUS_FRACTION,
    "update_clip_norm": UPDATE_CLIP_NORM, "grad_clip_norm": GRAD_CLIP_NORM,
}
with open(RESULTS / "master_config.json", "w") as f:
    json.dump(config, f, indent=2)

print(f"\n✅ All remaining experiments done. Results at {RESULTS}")


MALICIOUS-CLIENT DETECTION
MCDR = 1.0000 | TCE = 0.9600

BACKDOOR / ASR
 seed  clean_F1  clean_AUROC  clean_AUPRC  Backdoor_ASR  n_malicious
   42  0.609674     0.813719     0.757487      0.632190            2
   43  0.600925     0.780827     0.709318      0.016739            2
   44  0.609693     0.759922     0.690425      0.139747            2
   45  0.630408     0.798347     0.726542      0.190321            2
   46  0.628563     0.786635     0.717696      0.248621            2

SYBIL ATTACK
 seed  sybil_n  Accuracy  Precision   Recall       F1  Macro_F1    AUROC    AUPRC
   42        3  0.398009   0.344041 1.000000 0.511950  0.363306 0.713606 0.618350
   43        3  0.735852   0.591023 0.530451 0.559101  0.685272 0.697418 0.657692
   44        3  0.688429   0.505310 0.627676 0.559885  0.659372 0.748511 0.672357
   45        3  0.389809   0.336622 0.960753 0.498561  0.359691 0.582508 0.489662
   46        3  0.652578   0.465810 0.683623 0.554079  0.634757 0.740826 0.630486

MEMBER